## 1. Settings

In [70]:
# libraries
import pandas as pd
import numpy as np
import scipy.stats
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

In [71]:
# pandas options
pd.set_option("display.max_columns", None)

In [72]:
# ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [73]:
# garbage collection
import gc
gc.enable()

## 2. FUNCTIONS

In [74]:
##### FUNCTION FOR COUNTING MISSINGS
def count_missings(data):
    total = data.isnull().sum().sort_values(ascending = False)
    percent = (data.isnull().sum() / data.isnull().count() * 100).sort_values(ascending = False)
    table = pd.concat([total, percent], axis = 1, keys = ["Total", "Percent"])
    table = table[table["Total"] > 0]
    return table

In [77]:
##### FUNCTION FOR CREATING FLAGS FOR MISSINGS
def create_null_flags(data, features = None):
    if features == None:
        features = data.columns
    for var in features:
        num_null = data[var].isnull() + 0
        if num_null.sum() > 0:
            data["ISNULL_" + str(var)] = num_null
    return data

In [78]:
##### FUNCTION FOR TREATING FACTORS
def treat_factors(data, method = "label"):
    
    # label encoding
    if method == "label":
        factors = [f for f in data.columns if data[f].dtype == "object"]
        for var in factors:
            data[var], _ = pd.factorize(data[var])
        
    # dummy encoding
    if method == "dummy":
        data = pd.get_dummies(data, drop_first = True)
    
    # dataset
    return data

In [79]:
##### FUNCTION FOR COMPUTING ACCEPT/REJECT RATIOS
def compute_accept_reject_ratio(data, lags = [1, 3, 5]):
    
    # preparations
    dec_prev = data[["SK_ID_CURR", "SK_ID_PREV", "DAYS_DECISION", "NAME_CONTRACT_STATUS"]]
    dec_prev["DAYS_DECISION"] = -dec_prev["DAYS_DECISION"]
    dec_prev = dec_prev.sort_values(by = ["SK_ID_CURR", "DAYS_DECISION"])
    dec_prev = pd.get_dummies(dec_prev)
     
    # compuatation
    for t in lags:
        
        # acceptance ratios
        tmp = dec_prev[["SK_ID_CURR", "NAME_CONTRACT_STATUS_Approved"]].groupby(["SK_ID_CURR"]).head(1)
        tmp = tmp.groupby(["SK_ID_CURR"], as_index = False).mean()
        tmp.columns = ["SK_ID_CURR", "APPROVE_RATIO_" + str(t)]
        data = data.merge(tmp, how = "left", on = "SK_ID_CURR")
        
        # rejection ratios
        tmp = dec_prev[["SK_ID_CURR", "NAME_CONTRACT_STATUS_Refused"]].groupby(["SK_ID_CURR"]).head(1)
        tmp = tmp.groupby(["SK_ID_CURR"], as_index = False).mean()
        tmp.columns = ["SK_ID_CURR", "REJECT_RATIO_" + str(t)]
        data = data.merge(tmp, how = "left", on = "SK_ID_CURR")
        
    # dataset
    return data

In [80]:
##### FUNCTION FOR AGGREGATING DATA
def aggregate_data(data, id_var, label = None):
    
    
    ### SEPARATE FEATURES
  
    # display info
    print("- Preparing the dataset...")

    # find factors
    data_factors = [f for f in data.columns if data[f].dtype == "object"]
    
    # partition subsets
    num_data = data[list(set(data.columns) - set(data_factors))]
    fac_data = data[[id_var] + data_factors]
    
    # display info
    num_facs = fac_data.shape[1] - 1
    num_nums = num_data.shape[1] - 1
    print("- Extracted %.0f factors and %.0f numerics..." % (num_facs, num_nums))

    # aggregate numerics
    if (num_nums > 0):
        print("- Aggregating numeric features...")
        num_data = num_data.groupby(id_var).agg(["mean", "std", "min", "max"])
        num_data.columns = ["_".join(col).strip() for col in num_data.columns.values]
        num_data = num_data.sort_index()

    # aggregate factors
    if (num_facs > 0):
        print("- Aggregating factor features...")
        fac_data = fac_data.groupby(id_var).agg([("mode",   lambda x: scipy.stats.mode(x)[0][0]),
                                                 ("unique", lambda x: x.nunique())])
        fac_data.columns = ["_".join(col).strip() for col in fac_data.columns.values]
        fac_data = fac_data.sort_index()


    ##### MERGER

    # merge numerics and factors
    if ((num_facs > 0) & (num_nums > 0)):
        agg_data = pd.concat([num_data, fac_data], axis = 1)
    
    # use factors only
    if ((num_facs > 0) & (num_nums == 0)):
        agg_data = fac_data
        
    # use numerics only
    if ((num_facs == 0) & (num_nums > 0)):
        agg_data = num_data
        

    ##### LAST STEPS

    # update labels
    if label != None:
        agg_data.columns = [label + "_" + str(col) for col in agg_data.columns]
    
    # impute zeros for SD
    #stdevs = agg_data.filter(like = "_std").columns
    #for var in stdevs:
    #    agg_data[var].fillna(0, inplace = True)

    # display info
    print("- Final dimensions:", agg_data.shape)
    
    # return dataset
    return agg_data

In [81]:
def downcast_dtypes(df):
    float_cols = [c for c in df if df[c].dtype == "float64"]
    int_cols =   [c for c in df if df[c].dtype in ["int64"]]

    df[float_cols] = df[float_cols].astype(np.float32)
    df[int_cols]   = df[int_cols].astype(np.int32)

    return df


## 3. Data Import

In [82]:
# import data
train = pd.read_csv("../data/raw/application_train.csv")
test  = pd.read_csv("../data/raw/application_test.csv")
buro  = pd.read_csv("../data/raw/bureau.csv")
bbal  = pd.read_csv("../data/raw/bureau_balance.csv")
prev  = pd.read_csv("../data/raw/previous_application.csv")
card  = pd.read_csv("../data/raw/credit_card_balance.csv")
poca  = pd.read_csv("../data/raw/POS_CASH_balance.csv")
inst  = pd.read_csv("../data/raw/installments_payments.csv")

In [83]:
# check dimensions
print("Application:", train.shape, test.shape)
print("Buro:", buro.shape)
print("Bbal:", bbal.shape)
print("Prev:", prev.shape)
print("Card:", card.shape)
print("Poca:", poca.shape)
print("Inst:", inst.shape)

Application: (307511, 122) (48744, 121)
Buro: (1716428, 17)
Bbal: (27299925, 3)
Prev: (1670214, 37)
Card: (3840312, 23)
Poca: (10001358, 8)
Inst: (13605401, 8)


In [84]:
# extract target
y = train[["SK_ID_CURR", "TARGET"]]
del train["TARGET"]

In [85]:
y.shape

(307511, 2)

In [86]:
y_test = test[["SK_ID_CURR"]]
y_test.shape

(48744, 1)

## 4. Data Preprocessing & Feature Engineering

### 4.1. For Application Data

In [87]:
# concatenate application data
appl = pd.concat([train, test])
# del train, test

In [88]:
appl.head()

,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,NaN,1,1,0,1,1,0,Laborers,1.0,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.083037,0.262949,0.139376,0.0247,0.0369,0.9722,0.6192,0.0143,0.00,0.0690,0.0833,0.1250,0.0369,0.0202,0.0190,0.0000,0.0000,0.0252,0.0383,0.9722,0.6341,0.0144,0.0000,0.0690,0.0833,0.1250,0.0377,0.022,0.0198,0.0,0.0,0.0250,0.0369,0.9722,0.6243,0.0144,0.00,0.0690,0.0833,0.1250,0.0375,0.0205,0.0193,0.0000,0.00,reg oper account,block of flats,0.0149,"Stone, brick",No,2.0,2.0,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,1,1,MONDAY,11,0,0,0,0,0,0,School,0.311267,0.622246,NaN,0.0959,0.0529,0.9851,0.7960,0.0605,0.08,0.0345,0.2917,0.3333,0.0130,0.0773,0.0549,0.0039,0.0098,0.0924,0.0538,0.9851,0.8040,0.0497,0.0806,0.0345,0.2917,0.3333,0.0128,0.079,0.0554,0.0,0.0,0.0968,0.0529,0.9851,0.7987,0.0608,0.08,0.0345,0.2917,0.3333,0.0132,0.0787,0.0558,0.0039,0.01,reg oper account,block of flats,0.0714,Block,No,1.0,0.0,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,1,1,0,Laborers,1.0,2,2,MONDAY,9,0,0,0,0,0,0,Government,NaN,0.555912,0.729567,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

In [ ]:
### FEATURE ENGINEERING

# income ratios
appl["CREDIT_BY_INCOME"]      = appl["AMT_CREDIT"]      / appl["AMT_INCOME_TOTAL"]
appl["ANNUITY_BY_INCOME"]     = appl["AMT_ANNUITY"]     / appl["AMT_INCOME_TOTAL"]
appl["GOODS_PRICE_BY_INCOME"] = appl["AMT_GOODS_PRICE"] / appl["AMT_INCOME_TOTAL"]
appl["INCOME_PER_PERSON"]     = appl["AMT_INCOME_TOTAL"] / appl["CNT_FAM_MEMBERS"]

# career ratio
appl["PERCENT_WORKED"] = appl["DAYS_EMPLOYED"] / appl["DAYS_BIRTH"]
appl["PERCENT_WORKED"][appl["PERCENT_WORKED"] < 0] = None

# number of adults
appl["CNT_ADULTS"] = appl["CNT_FAM_MEMBERS"] - appl["CNT_CHILDREN"]
appl['CHILDREN_RATIO'] = appl['CNT_CHILDREN'] / appl['CNT_FAM_MEMBERS']

# number of overall payments
appl['ANNUITY LENGTH'] = appl['AMT_CREDIT'] / appl['AMT_ANNUITY']

# external sources
#appl["EXT_SOURCE_MIN"]  = appl[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].min(axis = 1)
#appl["EXT_SOURCE_MAX"]  = appl[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].max(axis = 1)
appl["EXT_SOURCE_MEAN"] = appl[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].mean(axis = 1)
#appl["EXT_SOURCE_SD"]   = appl[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].std(axis = 1)
appl["NUM_EXT_SOURCES"] = 3 - (appl["EXT_SOURCE_1"].isnull().astype(int) +
                               appl["EXT_SOURCE_2"].isnull().astype(int) +
                               appl["EXT_SOURCE_3"].isnull().astype(int))

# number of documents
doc_vars = ["FLAG_DOCUMENT_2",  "FLAG_DOCUMENT_3",  "FLAG_DOCUMENT_4",  "FLAG_DOCUMENT_5",  "FLAG_DOCUMENT_6",
            "FLAG_DOCUMENT_7",  "FLAG_DOCUMENT_8",  "FLAG_DOCUMENT_9",  "FLAG_DOCUMENT_10", "FLAG_DOCUMENT_11",
            "FLAG_DOCUMENT_12", "FLAG_DOCUMENT_13", "FLAG_DOCUMENT_14", "FLAG_DOCUMENT_15", "FLAG_DOCUMENT_16",
            "FLAG_DOCUMENT_17", "FLAG_DOCUMENT_18", "FLAG_DOCUMENT_19", "FLAG_DOCUMENT_20", "FLAG_DOCUMENT_21"]
appl["NUM_DOCUMENTS"] = appl[doc_vars].sum(axis = 1)

# application date
appl["DAY_APPR_PROCESS_START"] = "Working day"
appl["DAY_APPR_PROCESS_START"][(appl["WEEKDAY_APPR_PROCESS_START"] == "SATURDAY") |
                               (appl["WEEKDAY_APPR_PROCESS_START"] == "SUNDAY")] = "Weekend"

# age ratios
appl["OWN_CAR_AGE_RATIO"] = appl["OWN_CAR_AGE"] / appl["DAYS_BIRTH"]
appl["DAYS_ID_PUBLISHED_RATIO"] = appl["DAYS_ID_PUBLISH"] / appl["DAYS_BIRTH"]
appl["DAYS_REGISTRATION_RATIO"] = appl["DAYS_REGISTRATION"] / appl["DAYS_BIRTH"]
appl["DAYS_LAST_PHONE_CHANGE_RATIO"] = appl["DAYS_LAST_PHONE_CHANGE"] / appl["DAYS_BIRTH"]


##### FEATURE REMOVAL
drops = ['APARTMENTS_MEDI', 'BASEMENTAREA_MEDI', 'COMMONAREA_MEDI', 'ELEVATORS_MEDI', 'ENTRANCES_MEDI', 
         'FLOORSMAX_MEDI', 'FLOORSMIN_MEDI', 'LANDAREA_MEDI', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI',
         'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAREA_MEDI','YEARS_BEGINEXPLUATATION_MEDI', 'YEARS_BUILD_MEDI',
         'APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'COMMONAREA_MODE','ELEVATORS_MODE', 'ENTRANCES_MODE', 
         'FLOORSMAX_MODE', 'FLOORSMIN_MODE', 'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE', 
         'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE', 'TOTALAREA_MODE',  'YEARS_BEGINEXPLUATATION_MODE']
appl = appl.drop(columns = drops)

In [90]:
# rename features
appl.columns = ["SK_ID_CURR"] + ["app_" + str(col) for col in appl.columns if col not in "SK_ID_CURR"]

In [91]:
# check data
appl.head()

,SK_ID_CURR,app_NAME_CONTRACT_TYPE,app_CODE_GENDER,app_FLAG_OWN_CAR,app_FLAG_OWN_REALTY,app_CNT_CHILDREN,app_AMT_INCOME_TOTAL,app_AMT_CREDIT,app_AMT_ANNUITY,app_AMT_GOODS_PRICE,app_NAME_TYPE_SUITE,app_NAME_INCOME_TYPE,app_NAME_EDUCATION_TYPE,app_NAME_FAMILY_STATUS,app_NAME_HOUSING_TYPE,app_REGION_POPULATION_RELATIVE,app_DAYS_BIRTH,app_DAYS_EMPLOYED,app_DAYS_REGISTRATION,app_DAYS_ID_PUBLISH,app_OWN_CAR_AGE,app_FLAG_MOBIL,app_FLAG_EMP_PHONE,app_FLAG_WORK_PHONE,app_FLAG_CONT_MOBILE,app_FLAG_PHONE,app_FLAG_EMAIL,app_OCCUPATION_TYPE,app_CNT_FAM_MEMBERS,app_REGION_RATING_CLIENT,app_REGION_RATING_CLIENT_W_CITY,app_WEEKDAY_APPR_PROCESS_START,app_HOUR_APPR_PROCESS_START,app_REG_REGION_NOT_LIVE_REGION,app_REG_REGION_NOT_WORK_REGION,app_LIVE_REGION_NOT_WORK_REGION,app_REG_CITY_NOT_LIVE_CITY,app_REG_CITY_NOT_WORK_CITY,app_LIVE_CITY_NOT_WORK_CITY,app_ORGANIZATION_TYPE,app_EXT_SOURCE_1,app_EXT_SOURCE_2,app_EXT_SOURCE_3,app_APARTMENTS_AVG,app_BASEMENTAREA_AVG,app_YEARS_BEGINEXPLUATATION_AVG,app_YEARS_BUILD_AVG,app_COMMONAREA_AVG,app_ELEVATORS_AVG,app_ENTRANCES_AVG,app_FLOORSMAX_AVG,app_FLOORSMIN_AVG,app_LANDAREA_AVG,app_LIVINGAPARTMENTS_AVG,app_LIVINGAREA_AVG,app_NONLIVINGAPARTMENTS_AVG,app_NONLIVINGAREA_AVG,app_YEARS_BUILD_MODE,app_FONDKAPREMONT_MODE,app_HOUSETYPE_MODE,app_WALLSMATERIAL_MODE,app_EMERGENCYSTATE_MODE,app_OBS_30_CNT_SOCIAL_CIRCLE,app_DEF_30_CNT_SOCIAL_CIRCLE,app_OBS_60_CNT_SOCIAL_CIRCLE,app_DEF_60_CNT_SOCIAL_CIRCLE,app_DAYS_LAST_PHONE_CHANGE,app_FLAG_DOCUMENT_2,app_FLAG_DOCUMENT_3,app_FLAG_DOCUMENT_4,app_FLAG_DOCUMENT_5,app_FLAG_DOCUMENT_6,app_FLAG_DOCUMENT_7,app_FLAG_DOCUMENT_8,app_FLAG_DOCUMENT_9,app_FLAG_DOCUMENT_10,app_FLAG_DOCUMENT_11,app_FLAG_DOCUMENT_12,app_FLAG_DOCUMENT_13,app_FLAG_DOCUMENT_14,app_FLAG_DOCUMENT_15,app_FLAG_DOCUMENT_16,app_FLAG_DOCUMENT_17,app_FLAG_DOCUMENT_18,app_FLAG_DOCUMENT_19,app_FLAG_DOCUMENT_20,app_FLAG_DOCUMENT_21,app_AMT_REQ_CREDIT_BUREAU_HOUR,app_AMT_REQ_CREDIT_BUREAU_DAY,app_AMT_REQ_CREDIT_BUREAU_WEEK,app_AMT_REQ_CREDIT_BUREAU_MON,app_AMT_REQ_CREDIT_BUREAU_QRT,app_AMT_REQ_CREDIT_BUREAU_YEAR,app_CREDIT_BY_INCOME,app_ANNUITY_BY_INCOME,app_GOODS_PRICE_BY_INCOME,app_INCOME_PER_PERSON,app_PERCENT_WORKED,app_CNT_ADULTS,app_CHILDREN_RATIO,app_ANNUITY LENGTH,app_EXT_SOURCE_MEAN,app_NUM_EXT_SOURCES,app_NUM_DOCUMENTS,app_DAY_APPR_PROCESS_START,app_OWN_CAR_AGE_RATIO,app_DAYS_ID_PUBLISHED_RATIO,app_DAYS_REGISTRATION_RATIO,app_DAYS_LAST_PHONE_CHANGE_RATIO
0,100002,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,NaN,1,1,0,1,1,0,Laborers,1.0,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.083037,0.262949,0.139376,0.0247,0.0369,0.9722,0.6192,0.0143,0.00,0.0690,0.0833,0.1250,0.0369,0.0202,0.0190,0.0000,0.0000,0.6341,reg oper account,block of flats,"Stone, brick",No,2.0,2.0,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,2.007889,0.121978,1.733333,202500.0,0.067329,1.0,0.0,16.461104,0.161787,3,1,Working day,NaN,0.224078,0.385583,0.119860
1,100003,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,1,1,MONDAY,11,0,0,0,0,0,0,School,0.311267,0.622246,NaN,0.0959,0.0529,0.9851,0.7960,0.0605,0.08,0.0345,0.2917,0.3333,0.0130,0.0773,0.0549,0.0039,0.0098,0.8040,reg oper account,block of flats,Block,No,1.0,0.0,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,4.790750,0.132217,4.183333,135000.0,0.070862,2.0,0.0,36.234085,0.466757,2,1,Working day,NaN,0.017358,0.070743,0.049389
2,100004,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,1,1,0,Laborers,1.0,2,2,MONDAY,9,0,0,0,0,0,0,Government,NaN,0.555912,0.729567,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

In [92]:
# count missings
nas = count_missings(appl)
nas.head()

,Total,Percent
app_COMMONAREA_AVG,248360,69.714109
app_NONLIVINGAPARTMENTS_AVG,246861,69.293343
app_FONDKAPREMONT_MODE,243092,68.235393
app_LIVINGAPARTMENTS_AVG,242979,68.203674
app_FLOORSMIN_AVG,241108,67.678489


### 4.2. Credit Bureau Data

#### 4.2.1. BBAL Data

In [93]:
train_bbal_score = pd.read_csv('../data/prepared/bbal_score_train.csv',index_col=0)
test_bbal_score = pd.read_csv('../data/prepared/bbal_score_test.csv',index_col=0)
bbal_score= pd.concat([train_bbal_score, test_bbal_score])
bbal_score.head()

,bbal_score
SK_ID_CURR,
100002,0.469142
100010,0.474240
100019,0.535118
100032,0.427840
100033,0.465838


In [94]:
### FEATURE ENGINEERING

# loan default score
bbal["NUM_STATUS"] = 0
bbal["NUM_STATUS"][bbal["STATUS"] == "X"] = None
bbal["NUM_STATUS"][bbal["STATUS"] == "1"] = 1
bbal["NUM_STATUS"][bbal["STATUS"] == "2"] = 2
bbal["NUM_STATUS"][bbal["STATUS"] == "3"] = 3
bbal["NUM_STATUS"][bbal["STATUS"] == "4"] = 4
bbal["NUM_STATUS"][bbal["STATUS"] == "5"] = 5
bbal["LOAN_SCORE"] = bbal["NUM_STATUS"] / (abs(bbal["MONTHS_BALANCE"]) + 1)
loan_score = bbal.groupby("SK_ID_BUREAU", as_index = False).LOAN_SCORE.sum()
del bbal["NUM_STATUS"]
del bbal["LOAN_SCORE"]

# dummy encoding for STATUS
bbal = pd.get_dummies(bbal, columns = ["STATUS"], prefix = "STATUS")

In [95]:
# count missings
nas = count_missings(bbal)
nas.head()

,Total,Percent


In [96]:
### AGGREGATIONS

# total month count
cnt_mon = bbal[["SK_ID_BUREAU", "MONTHS_BALANCE"]].groupby("SK_ID_BUREAU").count()
del bbal["MONTHS_BALANCE"]

# aggregate data
agg_bbal = bbal.groupby("SK_ID_BUREAU").mean()

# add total month count
agg_bbal["MONTH_COUNT"] = cnt_mon

# add loan score
agg_bbal = agg_bbal.merge(loan_score, how = "left", on = "SK_ID_BUREAU")


In [97]:
# count missings
nas = count_missings(agg_bbal)
nas.head()

,Total,Percent


In [98]:
# check data
agg_bbal.head()

,SK_ID_BUREAU,STATUS_0,STATUS_1,STATUS_2,STATUS_3,STATUS_4,STATUS_5,STATUS_C,STATUS_X,MONTH_COUNT,LOAN_SCORE
0,5001709,0.000000,0.0,0.0,0.0,0.0,0.0,0.886598,0.113402,97,0.0
1,5001710,0.060241,0.0,0.0,0.0,0.0,0.0,0.578313,0.361446,83,0.0
2,5001711,0.750000,0.0,0.0,0.0,0.0,0.0,0.000000,0.250000,4,0.0
3,5001712,0.526316,0.0,0.0,0.0,0.0,0.0,0.473684,0.000000,19,0.0
4,5001713,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,1.000000,22,0.0


In [99]:
# clear memory
del bbal

#### 4.2.2. BURO Data

In [100]:
# check buro data
buro.head()

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


In [101]:
### 把 agg_bbal 合并到 buro 
buro = buro.merge(right = agg_bbal.reset_index(), how = "left", on = "SK_ID_BUREAU")

In [ ]:
##### FEATURE ENGINEERING

# number of buro loans 
cnt_buro = buro[["SK_ID_CURR", "SK_ID_BUREAU"]].groupby(["SK_ID_CURR"], as_index = False).count()
cnt_buro.columns = ["SK_ID_CURR", "CNT_BURO_LOANS"]
buro = buro.merge(cnt_buro, how = "left", on = "SK_ID_CURR")

# amount ratios
buro["AMT_SUM_OVERDUE_RATIO_1"] = buro["AMT_CREDIT_SUM_OVERDUE"] / buro["AMT_ANNUITY"]
buro["AMT_SUM_OVERDUE_RATIO_2"] = buro["AMT_CREDIT_SUM_OVERDUE"] / buro["AMT_CREDIT_SUM"]
buro["AMT_MAX_OVERDUE_RATIO_1"] = buro["AMT_CREDIT_MAX_OVERDUE"] / buro["AMT_ANNUITY"]
buro["AMT_MAX_OVERDUE_RATIO_2"] = buro["AMT_CREDIT_MAX_OVERDUE"] / buro["AMT_CREDIT_SUM"]
buro["AMT_SUM_DEBT_RATIO_1"]    = buro["AMT_CREDIT_SUM_DEBT"] / buro["AMT_CREDIT_SUM"]
buro["AMT_SUM_DEBT_RATIO_2"]    = buro["AMT_CREDIT_SUM_DEBT"] / buro["AMT_CREDIT_SUM_LIMIT"]

# recency-weighted loan score
buro["WEIGHTED_LOAN_SCORE"] = buro["LOAN_SCORE"] / (buro["DAYS_CREDIT"] / 12)

# day differences
buro["DAYS_END_DIFF_1"] = buro["DAYS_ENDDATE_FACT"]   - buro["DAYS_CREDIT_ENDDATE"]
buro["DAYS_END_DIFF_2"] = buro["DAYS_CREDIT_UPDATE"]  - buro["DAYS_CREDIT_ENDDATE"]
buro["DAYS_DURATION_1"] = buro["DAYS_CREDIT_ENDDATE"] - buro["DAYS_CREDIT"]
buro["DAYS_DURATION_2"] = buro["DAYS_ENDDATE_FACT"]   - buro["DAYS_CREDIT"]

# number of active buro loans
cnt_buro = buro[["SK_ID_CURR", "CREDIT_ACTIVE"]]
cnt_buro.columns = ["SK_ID_CURR", "CNT_BURO_ACTIVE"]
cnt_buro = cnt_buro[cnt_buro["CNT_BURO_ACTIVE"] == "Active"]
cnt_buro = cnt_buro[["SK_ID_CURR", "CNT_BURO_ACTIVE"]].groupby(["SK_ID_CURR"], as_index = False).count()
buro = buro.merge(cnt_buro, how = "left", on = "SK_ID_CURR")
buro["CNT_BURO_ACTIVE"].fillna(0, inplace = True)

# number of closed buro loans
cnt_buro = buro[["SK_ID_CURR", "CREDIT_ACTIVE"]]
cnt_buro.columns = ["SK_ID_CURR", "CNT_BURO_CLOSED"]
cnt_buro = cnt_buro[cnt_buro["CNT_BURO_CLOSED"] == "Closed"]
cnt_buro = cnt_buro[["SK_ID_CURR", "CNT_BURO_CLOSED"]].groupby(["SK_ID_CURR"], as_index = False).count()
buro = buro.merge(cnt_buro, how = "left", on = "SK_ID_CURR")
buro["CNT_BURO_CLOSED"].fillna(0, inplace = True)

# number of defaulted buro loans
cnt_buro = buro[["SK_ID_CURR", "CREDIT_ACTIVE"]]
cnt_buro.columns = ["SK_ID_CURR", "CNT_BURO_BAD"]
cnt_buro = cnt_buro[cnt_buro["CNT_BURO_BAD"] == "Bad debt"]
cnt_buro = cnt_buro[["SK_ID_CURR", "CNT_BURO_BAD"]].groupby(["SK_ID_CURR"], as_index = False).count()
buro = buro.merge(cnt_buro, how = "left", on = "SK_ID_CURR")
buro["CNT_BURO_BAD"].fillna(0, inplace = True)

In [103]:
# dummy encodnig for factors
buro = pd.get_dummies(buro, drop_first = True)

In [104]:
# count missings
nas = count_missings(buro)
nas.head()

,Total,Percent
AMT_MAX_OVERDUE_RATIO_1,1629591,94.940831
AMT_SUM_OVERDUE_RATIO_1,1483326,86.419355
AMT_SUM_DEBT_RATIO_2,1336100,77.841890
AMT_ANNUITY,1226791,71.473490
AMT_MAX_OVERDUE_RATIO_2,1149800,66.987954


In [105]:
### AGGREGATIONS

# count previous buro loans
cnt_buro = buro[["SK_ID_CURR", "SK_ID_BUREAU"]].groupby("SK_ID_CURR").count()
del buro["SK_ID_BUREAU"]

# aggregate data
agg_buro = aggregate_data(buro, id_var = "SK_ID_CURR", label = "buro")

# add buro loan count
agg_buro["buro_BURO_COUNT"] = cnt_buro

# clean up
omits = ["WEIGHTED_LOAN_SCORE"]
for var in omits:
    del agg_buro["buro_" + str(var) + "_std"]
    del agg_buro["buro_" + str(var) + "_min"]
    del agg_buro["buro_" + str(var) + "_max"]

- Preparing the dataset...
- Extracted 0 factors and 58 numerics...
- Aggregating numeric features...
- Final dimensions: (305811, 232)


In [106]:
# count missings
nas = count_missings(agg_buro)
nas.head()

,Total,Percent
buro_AMT_SUM_DEBT_RATIO_2_std,302007,98.756094
buro_AMT_MAX_OVERDUE_RATIO_1_std,288362,94.294188
buro_AMT_MAX_OVERDUE_RATIO_1_min,253792,82.989821
buro_AMT_MAX_OVERDUE_RATIO_1_max,253792,82.989821
buro_AMT_MAX_OVERDUE_RATIO_1_mean,253792,82.989821


In [107]:
# Add BURO Score generated by GRU
agg_buro = agg_buro.merge(right = bbal_score.reset_index(), how = "left", on = "SK_ID_CURR")
agg_buro.head()

,SK_ID_CURR,buro_CNT_BURO_CLOSED_mean,buro_CNT_BURO_CLOSED_std,buro_CNT_BURO_CLOSED_min,buro_CNT_BURO_CLOSED_max,buro_CREDIT_TYPE_Consumer credit_mean,buro_CREDIT_TYPE_Consumer credit_std,buro_CREDIT_TYPE_Consumer credit_min,buro_CREDIT_TYPE_Consumer credit_max,buro_STATUS_1_mean,buro_STATUS_1_std,buro_STATUS_1_min,buro_STATUS_1_max,buro_CREDIT_CURRENCY_currency 4_mean,buro_CREDIT_CURRENCY_currency 4_std,buro_CREDIT_CURRENCY_currency 4_min,buro_CREDIT_CURRENCY_currency 4_max,buro_CREDIT_TYPE_Real estate loan_mean,buro_CREDIT_TYPE_Real estate loan_std,buro_CREDIT_TYPE_Real estate loan_min,buro_CREDIT_TYPE_Real estate loan_max,buro_AMT_CREDIT_SUM_mean,buro_AMT_CREDIT_SUM_std,buro_AMT_CREDIT_SUM_min,buro_AMT_CREDIT_SUM_max,buro_CREDIT_TYPE_Credit card_mean,buro_CREDIT_TYPE_Credit card_std,buro_CREDIT_TYPE_Credit card_min,buro_CREDIT_TYPE_Credit card_max,buro_AMT_MAX_OVERDUE_RATIO_1_mean,buro_AMT_MAX_OVERDUE_RATIO_1_std,buro_AMT_MAX_OVERDUE_RATIO_1_min,buro_AMT_MAX_OVERDUE_RATIO_1_max,buro_CREDIT_TYPE_Interbank credit_mean,buro_CREDIT_TYPE_Interbank credit_std,buro_CREDIT_TYPE_Interbank credit_min,buro_CREDIT_TYPE_Interbank credit_max,buro_DAYS_CREDIT_mean,buro_DAYS_CREDIT_std,buro_DAYS_CREDIT_min,buro_DAYS_CREDIT_max,buro_CREDIT_TYPE_Mobile operator loan_mean,buro_CREDIT_TYPE_Mobile operator loan_std,buro_CREDIT_TYPE_Mobile operator loan_min,buro_CREDIT_TYPE_Mobile operator loan_max,buro_CREDIT_TYPE_Unknown type of loan_mean,buro_CREDIT_TYPE_Unknown type of loan_std,buro_CREDIT_TYPE_Unknown type of loan_min,buro_CREDIT_TYPE_Unknown type of loan_max,buro_STATUS_2_mean,buro_STATUS_2_std,buro_STATUS_2_min,buro_STATUS_2_max,buro_CREDIT_TYPE_Loan for purchase of shares (margin lending)_mean,buro_CREDIT_TYPE_Loan for purchase of shares (margin lending)_std,buro_CREDIT_TYPE_Loan for purchase of shares (margin lending)_min,buro_CREDIT_TYPE_Loan for purchase of shares (margin lending)_max,buro_CREDIT_DAY_OVERDUE_mean,buro_CREDIT_DAY_OVERDUE_std,buro_CREDIT_DAY_OVERDUE_min,buro_CREDIT_DAY_OVERDUE_max,buro_CREDIT_TYPE_Loan for working capital replenishment_mean,buro_CREDIT_TYPE_Loan for working capital replenishment_std,buro_CREDIT_TYPE_Loan for working capital replenishment_min,buro_CREDIT_TYPE_Loan for working capital replenishment_max,buro_index_mean,buro_index_std,buro_index_min,buro_index_max,buro_CREDIT_ACTIVE_Bad debt_mean,buro_CREDIT_ACTIVE_Bad debt_std,buro_CREDIT_ACTIVE_Bad debt_min,buro_CREDIT_ACTIVE_Bad debt_max,buro_AMT_SUM_OVERDUE_RATIO_1_mean,buro_AMT_SUM_OVERDUE_RATIO_1_std,buro_AMT_SUM_OVERDUE_RATIO_1_min,buro_AMT_SUM_OVERDUE_RATIO_1_max,buro_STATUS_3_mean,buro_STATUS_3_std,buro_STATUS_3_min,buro_STATUS_3_max,buro_AMT_CREDIT_SUM_DEBT_mean,buro_AMT_CREDIT_SUM_DEBT_std,buro_AMT_CREDIT_SUM_DEBT_min,buro_AMT_CREDIT_SUM_DEBT_max,buro_AMT_SUM_OVERDUE_RATIO_2_mean,buro_AMT_SUM_OVERDUE_RATIO_2_std,buro_AMT_SUM_OVERDUE_RATIO_2_min,buro_AMT_SUM_OVERDUE_RATIO_2_max,buro_DAYS_END_DIFF_1_mean,buro_DAYS_END_DIFF_1_std,buro_DAYS_END_DIFF_1_min,buro_DAYS_END_DIFF_1_max,buro_DAYS_DURATION_2_mean,buro_DAYS_DURATION_2_std,buro_DAYS_DURATION_2_min,buro_DAYS_DURATION_2_max,buro_CNT_BURO_ACTIVE_mean,buro_CNT_BURO_ACTIVE_std,buro_CNT_BURO_ACTIVE_min,buro_CNT_BURO_ACTIVE_max,buro_AMT_CREDIT_SUM_OVERDUE_mean,buro_AMT_CREDIT_SUM_OVERDUE_std,buro_AMT_CREDIT_SUM_OVERDUE_min,buro_AMT_CREDIT_SUM_OVERDUE_max,buro_AMT_SUM_DEBT_RATIO_1_mean,buro_AMT_SUM_DEBT_RATIO_1_std,buro_AMT_SUM_DEBT_RATIO_1_min,buro_AMT_SUM_DEBT_RATIO_1_max,buro_CREDIT_CURRENCY_currency 2_mean,buro_CREDIT_CURRENCY_currency 2_std,buro_CREDIT_CURRENCY_currency 2_min,buro_CREDIT_CURRENCY_currency 2_max,buro_DAYS_CREDIT_ENDDATE_mean,buro_DAYS_CREDIT_ENDDATE_std,buro_DAYS_CREDIT_ENDDATE_min,buro_DAYS_CREDIT_ENDDATE_max,buro_CREDIT_TYPE_Mortgage_mean,buro_CREDIT_TYPE_Mortgage_std,buro_CREDIT_TYPE_Mortgage_min,buro_CREDIT_TYPE_Mortgage_max,buro_AMT_SUM_DEBT_RATIO_2_mean,buro_AMT_SUM_DEBT_RATIO_2_std,buro_AMT_SUM_DEBT_RATIO_2_min,buro_AMT_SUM_DEBT_RATIO_2_max,buro_STATUS_4_mean,buro_STATUS_4_

In [108]:
# clear memory
del buro

### 4.3. Previous Loan Data

#### 4.3.1. INST Data

In [109]:
# check inst data
inst.head()

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585


In [110]:
### Score File
train_inst_score = pd.read_csv('../data/prepared/inst_score_train.csv',index_col=0)
test_inst_score = pd.read_csv('../data/prepared/inst_score_test.csv',index_col=0)
inst_score= pd.concat([train_inst_score, test_inst_score])

In [111]:
inst_prev_last = inst.groupby('SK_ID_PREV')['AMT_PAYMENT'].sum()
####

inst_NUM_INSTALMENT_VERSION = inst.groupby(['SK_ID_CURR'])['NUM_INSTALMENT_VERSION'].nunique()

#merge payments of same month
#maybe helpful for: inst.loc[(inst.SK_ID_PREV==1000005) & (inst.SK_ID_CURR==176456) & (inst.NUM_INSTALMENT_NUMBER==9)]
inst['DAYS_ENTRY_PAYMENT_weighted'] = inst['DAYS_ENTRY_PAYMENT'] * inst['AMT_PAYMENT']
inst = inst.groupby(['SK_ID_PREV','SK_ID_CURR','NUM_INSTALMENT_NUMBER']).agg({'DAYS_INSTALMENT':'mean',
                                                                       'DAYS_ENTRY_PAYMENT_weighted':'sum',
                                                                       'AMT_INSTALMENT':'mean',
                                                                       'AMT_PAYMENT':'sum'})
inst['DAYS_ENTRY_PAYMENT'] = inst['DAYS_ENTRY_PAYMENT_weighted']/inst['AMT_PAYMENT']
inst = inst.reset_index()
del inst['DAYS_ENTRY_PAYMENT_weighted']

inst_target1 = inst.loc[(inst['DAYS_ENTRY_PAYMENT']>inst['DAYS_INSTALMENT']+1)|(inst['AMT_PAYMENT']<inst['AMT_INSTALMENT'])].SK_ID_PREV.unique()

#create some new feature: how many days payment delayed? how much overpayed/underpayed?
inst['AMT_PAYMENT_PERC'] = inst['AMT_PAYMENT'] / inst['AMT_INSTALMENT']
inst['DPD'] = inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']
inst['DBD'] = inst['DAYS_INSTALMENT'] - inst['DAYS_ENTRY_PAYMENT']
inst['DPD'] = inst['DPD'].apply(lambda x: x if x > 0 else 0)
inst['DBD'] = inst['DBD'].apply(lambda x: x if x > 0 else 0)
inst['DPD'].fillna(30, inplace=True)
inst['DBD'].fillna(0, inplace=True)
inst['AMT_PAYMENT_DIFF'] = inst['AMT_INSTALMENT'] - inst['AMT_PAYMENT']
inst['DAYS_ENTRY_PAYMENT_SCALE'] = (inst['DAYS_ENTRY_PAYMENT']/365.25).apply(np.exp)
inst['DPD_SCALE'] = inst['DPD'] * inst['DAYS_ENTRY_PAYMENT_SCALE']
inst['DBD_SCALE'] = inst['DBD'] * inst['DAYS_ENTRY_PAYMENT_SCALE']
inst['AMT_PAYMENT_DIFF_SCALE'] = inst['AMT_PAYMENT_DIFF'] * inst['DAYS_ENTRY_PAYMENT_SCALE']
inst['AMT_PAYMENT_SCALE'] = inst['AMT_PAYMENT'] * inst['DAYS_ENTRY_PAYMENT_SCALE']

#max
inst_max = inst.groupby('SK_ID_CURR')[['DPD','DBD','AMT_PAYMENT_DIFF','AMT_PAYMENT_PERC']].max()
inst_max.columns = ['max_' + f_ for f_ in inst_max.columns]

#var
inst_var = inst.groupby('SK_ID_CURR')[['DPD','DBD','AMT_PAYMENT_DIFF','AMT_PAYMENT_PERC']].var()
inst_var.columns = ['var_' + f_ for f_ in inst_var.columns]

#sum
inst_sum = inst.groupby('SK_ID_CURR')[['DPD_SCALE','DBD_SCALE','AMT_PAYMENT_DIFF_SCALE','AMT_PAYMENT_SCALE']].sum()

#time-scaled mean
inst_day_scale_sum = inst.groupby('SK_ID_CURR')['DAYS_ENTRY_PAYMENT_SCALE'].sum()
inst_avg_scale = pd.DataFrame()
for f_ in inst_sum.columns:
    inst_avg_scale[f_] = inst_sum[f_]/inst_day_scale_sum
    
inst_sum.columns = ['sum_' + f_ for f_ in inst_sum.columns]
inst_avg_scale.columns = ['mean_' + f_ for f_ in inst_avg_scale.columns]

inst_avg = inst.groupby('SK_ID_CURR')[['DPD','DBD','AMT_PAYMENT_DIFF','AMT_PAYMENT','AMT_PAYMENT_PERC']].mean()
inst_avg.columns = ['mean_' + f_ for f_ in inst_avg.columns]

#when is the last time late
inst_last_late = inst[inst.DAYS_INSTALMENT < inst.DAYS_ENTRY_PAYMENT].groupby(['SK_ID_CURR'])['DAYS_INSTALMENT'].max()
inst_last_late.rename('DAYS_LAST_LATE',inplace=True)

#when is the last time underpaid
inst_last_underpaid = inst[inst.AMT_INSTALMENT < inst.AMT_PAYMENT].groupby(['SK_ID_CURR'])['DAYS_INSTALMENT'].max()
inst_last_underpaid.rename('DAYS_LAST_UNDERPAID',inplace=True)

#merge
inst_avg = inst_avg.merge(inst_max, on='SK_ID_CURR', how='outer')
inst_avg = inst_avg.merge(inst_var, on='SK_ID_CURR', how='outer')
inst_avg = inst_avg.merge(inst_sum, on='SK_ID_CURR', how='outer')
inst_avg = inst_avg.merge(inst_avg_scale, on='SK_ID_CURR', how='outer')
inst_avg['DAYS_LAST_LATE'] = inst_last_late
inst_avg['DAYS_LAST_UNDERPAID'] = inst_last_underpaid
inst_avg['N_NUM_INSTALMENT_VERSION'] = inst_NUM_INSTALMENT_VERSION
inst_avg['AMT_PAYMENT_TOTAL_RATIO'] = inst.groupby('SK_ID_CURR')['AMT_PAYMENT'].sum()/inst.groupby('SK_ID_CURR')['AMT_INSTALMENT'].sum()

inst_avg['length'] = inst[['SK_ID_CURR', 'SK_ID_PREV']].groupby('SK_ID_CURR').count()
inst_avg['count'] = inst[['SK_ID_CURR', 'SK_ID_PREV']].groupby('SK_ID_CURR')['SK_ID_PREV'].nunique()
inst_avg.columns = ['inst_' + f_ for f_ in inst_avg.columns]
inst_avg = downcast_dtypes(inst_avg)

#del inst, inst_sum, inst_max, inst_var
gc.collect()
inst_avg.head()

,inst_mean_DPD,inst_mean_DBD,inst_mean_AMT_PAYMENT_DIFF,inst_mean_AMT_PAYMENT,inst_mean_AMT_PAYMENT_PERC,inst_max_DPD,inst_max_DBD,inst_max_AMT_PAYMENT_DIFF,inst_max_AMT_PAYMENT_PERC,inst_var_DPD,inst_var_DBD,inst_var_AMT_PAYMENT_DIFF,inst_var_AMT_PAYMENT_PERC,inst_sum_DPD_SCALE,inst_sum_DBD_SCALE,inst_sum_AMT_PAYMENT_DIFF_SCALE,inst_sum_AMT_PAYMENT_SCALE,inst_mean_DPD_SCALE,inst_mean_DBD_SCALE,inst_mean_AMT_PAYMENT_DIFF_SCALE,inst_mean_AMT_PAYMENT_SCALE,inst_DAYS_LAST_LATE,inst_DAYS_LAST_UNDERPAID,inst_N_NUM_INSTALMENT_VERSION,inst_AMT_PAYMENT_TOTAL_RATIO,inst_length,inst_count
SK_ID_CURR,,,,,,,,,,,,,,,,,,,,,,,,,,,
100001,1.571429,8.857142,0.0,5885.132324,1.0,11.0,36.0,0.0,1.0,17.285715,164.142853,0.0,0.0,0.004197,0.604950,0.0,320.366364,0.100849,14.536783,0.0,7698.318359,-2886.0,NaN,2,1.0,7,2
100002,0.000000,20.421053,0.0,11559.247070,1.0,0.0,31.0,0.0,1.0,0.000000,24.257311,0.0,0.0,0.000000,169.398758,0.0,120482.359375,0.000000,19.079042,0.0,13569.686523,NaN,NaN,2,1.0,19,1
100003,0.000000,7.160000,0.0,64754.585938,1.0,0.0,14.0,0.0,1.0,0.000000,13.890000,0.0,0.0,0.000000,15.284735,0.0,281154.312500,0.000000,7.242818,0.0,133227.671875,NaN,NaN,2,1.0,25,3
100004,0.000000,7.666667,0.0,7096.154785,1.0,0.0,11.0,0.0,1.0,0.000000,17.333334,0.0,0.0,0.000000,2.771938,0.0,2715.767822,0.000000,7.413993,0.0,7263.757324,NaN,NaN,2,1.0,3,1
100005,0.111111,23.666666,0.0,6240.205078,1.0,1.0,37.0,0.0,1.0,0.111111,176.500000,0.0,0.0,0.201565,37.750141,0.0,11936.158203,0.115642,21.657967,0.0,6847.998047,-586.0,NaN,2,1.0,9,1


In [112]:
# Add INST Score generated by GRU
agg_inst = inst_avg.merge(right = inst_score.reset_index(), how = "left", on = "SK_ID_CURR")
agg_inst.head()


,SK_ID_CURR,inst_mean_DPD,inst_mean_DBD,inst_mean_AMT_PAYMENT_DIFF,inst_mean_AMT_PAYMENT,inst_mean_AMT_PAYMENT_PERC,inst_max_DPD,inst_max_DBD,inst_max_AMT_PAYMENT_DIFF,inst_max_AMT_PAYMENT_PERC,inst_var_DPD,inst_var_DBD,inst_var_AMT_PAYMENT_DIFF,inst_var_AMT_PAYMENT_PERC,inst_sum_DPD_SCALE,inst_sum_DBD_SCALE,inst_sum_AMT_PAYMENT_DIFF_SCALE,inst_sum_AMT_PAYMENT_SCALE,inst_mean_DPD_SCALE,inst_mean_DBD_SCALE,inst_mean_AMT_PAYMENT_DIFF_SCALE,inst_mean_AMT_PAYMENT_SCALE,inst_DAYS_LAST_LATE,inst_DAYS_LAST_UNDERPAID,inst_N_NUM_INSTALMENT_VERSION,inst_AMT_PAYMENT_TOTAL_RATIO,inst_length,inst_count,inst_score
0,100001,1.571429,8.857142,0.0,5885.132324,1.0,11.0,36.0,0.0,1.0,17.285715,164.142853,0.0,0.0,0.004197,0.604950,0.0,320.366364,0.100849,14.536783,0.0,7698.318359,-2886.0,NaN,2,1.0,7,2,0.490180
1,100002,0.000000,20.421053,0.0,11559.247070,1.0,0.0,31.0,0.0,1.0,0.000000,24.257311,0.0,0.0,0.000000,169.398758,0.0,120482.359375,0.000000,19.079042,0.0,13569.686523,NaN,NaN,2,1.0,19,1,0.475274
2,100003,0.000000,7.160000,0.0,64754.585938,1.0,0.0,14.0,0.0,1.0,0.000000,13.890000,0.0,0.0,0.000000,15.284735,0.0,281154.312500,0.000000,7.242818,0.0,133227.671875,NaN,NaN,2,1.0,25,3,0.363200
3,100004,0.000000,7.666667,0.0,7096.154785,1.0,0.0,11.0,0.0,1.0,0.000000,17.333334,0.0,0.0,0.000000,2.771938,0.0,2715.767822,0.000000,7.413993,0.0,7263.757324,NaN,NaN,2,1.0,3,1,0.516723
4,100005,0.111111,23.666666,0.0,6240.205078,1.0,1.0,37.0,0.0,1.0,0.111111,176.500000,0.0,0.0,0.201565,37.750141,0.0,11936.158203,0.115642,21.657967,0.0,6847.998047,-586.0,NaN,2,1.0,9,1,0.517884


#### 4.3.2 POCA Data

In [113]:
poca.head()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0


In [114]:
### Score file
train_pos_score = pd.read_csv('../data/prepared/pos_score_train.csv',index_col=0)
test_pos_score = pd.read_csv('../data/prepared/pos_score_test.csv',index_col=0)
poca_score= pd.concat([train_pos_score, test_pos_score])


In [115]:
poca_target1 = poca.SK_ID_PREV.loc[poca.SK_DPD_DEF > 0].unique()

# later use with prev
idx = poca.groupby(['SK_ID_PREV'])['MONTHS_BALANCE'].idxmax()  # most recent data
poca_prev_last = poca[['SK_ID_PREV', 'CNT_INSTALMENT', 'CNT_INSTALMENT_FUTURE']].loc[idx.values]
poca_prev_last['INSTAL_LEFT_RATIO'] = poca_prev_last['CNT_INSTALMENT_FUTURE'] / (poca_prev_last['CNT_INSTALMENT'])
poca_prev_last.set_index('SK_ID_PREV', inplace=True)

#####

idx = poca.groupby(['SK_ID_CURR'])['MONTHS_BALANCE'].idxmax()  # most recent data
poca_recent = poca[['SK_ID_CURR', 'MONTHS_BALANCE', 'CNT_INSTALMENT', 'CNT_INSTALMENT_FUTURE',
                    'NAME_CONTRACT_STATUS', 'SK_DPD', 'SK_DPD_DEF']].loc[idx.values]
poca_recent['NAME_CONTRACT_STATUS'], indexer = pd.factorize(poca_recent['NAME_CONTRACT_STATUS'])
poca_recent.set_index('SK_ID_CURR', inplace=True)
poca_recent.columns = ['recent_' + f_ for f_ in poca_recent.columns]

NAME_CONTRACT_STATUS_COUNT = pd.Series(poca.groupby(['SK_ID_CURR'])['NAME_CONTRACT_STATUS'].value_counts(),
                                       name='NAME_CONTRACT_STATUS_COUNT')
NAME_CONTRACT_STATUS_COUNT = pd.pivot_table(NAME_CONTRACT_STATUS_COUNT.reset_index(),
                                            index='SK_ID_CURR', columns='NAME_CONTRACT_STATUS',
                                            values='NAME_CONTRACT_STATUS_COUNT', fill_value=0)
NAME_CONTRACT_STATUS_COUNT.columns = ['NAME_CONTRACT_STATUS_CNT_' + f_ for f_ in NAME_CONTRACT_STATUS_COUNT.columns]

# aggregate features
poca['YEAR_SCALE'] = (poca['MONTHS_BALANCE'] / 12.0).apply(np.exp)
poca['SK_DPD_SCALE'] = poca['SK_DPD'] * poca['YEAR_SCALE']
poca['SK_DPD_DEF_SCALE'] = poca['SK_DPD_DEF'] * poca['YEAR_SCALE']

poca_max = poca.groupby(['SK_ID_CURR'])[['SK_DPD', 'SK_DPD_DEF']].max()
poca_max.columns = ['max_' + f_ for f_ in poca_max.columns]

poca_mean = poca.groupby(['SK_ID_CURR'])[['SK_DPD', 'SK_DPD_DEF']].mean()
poca_mean.columns = ['mean_' + f_ for f_ in poca_mean.columns]

poca_sum = poca.groupby(['SK_ID_CURR'])[['SK_DPD_SCALE', 'SK_DPD_DEF_SCALE']].sum()

poca_year_sum = poca.groupby(['SK_ID_CURR'])['YEAR_SCALE'].sum()
poca_mean_scale = pd.DataFrame()
for f_ in poca_sum.columns:
    poca_mean_scale[f_] = poca_sum[f_] / poca_year_sum

poca_sum.columns = ['sum_' + f_ for f_ in poca_sum.columns]
poca_mean_scale.columns = ['mean_' + f_ for f_ in poca_mean_scale.columns]

# what is the last month with DPD
poca_last_DPD = poca[poca.SK_DPD > 0].groupby(['SK_ID_CURR'])['MONTHS_BALANCE'].max()
poca_last_DPD.rename('MONTH_LAST_DPD', inplace=True)

# merge to poca table
poca_recent = poca_recent.merge(poca_max, how='outer', on='SK_ID_CURR')
poca_recent = poca_recent.merge(poca_mean, how='outer', on='SK_ID_CURR')
poca_recent = poca_recent.merge(poca_sum, how='outer', on='SK_ID_CURR')
poca_recent = poca_recent.merge(poca_mean_scale, how='outer', on='SK_ID_CURR')
poca_recent['MONTH_LAST_DPD'] = poca_last_DPD
poca_recent = poca_recent.merge(NAME_CONTRACT_STATUS_COUNT, how='outer', on='SK_ID_CURR')
poca_recent['MONTH_CNT'] = poca.groupby('SK_ID_CURR')['MONTHS_BALANCE'].count()
poca_recent['MONTH_MAX'] = poca.groupby('SK_ID_CURR')['MONTHS_BALANCE'].min()
poca_recent['count'] = poca.groupby('SK_ID_CURR')['SK_ID_PREV'].nunique()

poca_recent.fillna(0, inplace=True)
poca_recent = downcast_dtypes(poca_recent)
poca_recent.columns = ['poca_' + f_ for f_ in poca_recent.columns]

#meanenc_feats.append('poca_recent_NAME_CONTRACT_STATUS')
#del poca, poca_max, poca_mean, poca_sum, poca_mean_scale
gc.collect()
poca_recent.head()

,poca_recent_MONTHS_BALANCE,poca_recent_CNT_INSTALMENT,poca_recent_CNT_INSTALMENT_FUTURE,poca_recent_NAME_CONTRACT_STATUS,poca_recent_SK_DPD,poca_recent_SK_DPD_DEF,poca_max_SK_DPD,poca_max_SK_DPD_DEF,poca_mean_SK_DPD,poca_mean_SK_DPD_DEF,poca_sum_SK_DPD_SCALE,poca_sum_SK_DPD_DEF_SCALE,poca_mean_SK_DPD_SCALE,poca_mean_SK_DPD_DEF_SCALE,poca_MONTH_LAST_DPD,poca_NAME_CONTRACT_STATUS_CNT_Active,poca_NAME_CONTRACT_STATUS_CNT_Amortized debt,poca_NAME_CONTRACT_STATUS_CNT_Approved,poca_NAME_CONTRACT_STATUS_CNT_Canceled,poca_NAME_CONTRACT_STATUS_CNT_Completed,poca_NAME_CONTRACT_STATUS_CNT_Demand,poca_NAME_CONTRACT_STATUS_CNT_Returned to the store,poca_NAME_CONTRACT_STATUS_CNT_Signed,poca_NAME_CONTRACT_STATUS_CNT_XNA,poca_MONTH_CNT,poca_MONTH_MAX,poca_count
SK_ID_CURR,,,,,,,,,,,,,,,,,,,,,,,,,,,
100001,-53,4.0,0.0,0,0,0,7,7,0.777778,0.777778,0.002552,0.002552,0.048169,0.048169,-95.0,7,0,0,0,2,0,0,0,0,9,-96,2
100002,-1,24.0,6.0,1,0,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,19,0,0,0,0,0,0,0,0,19,-19,1
100003,-18,7.0,0.0,0,0,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,26,0,0,0,2,0,0,0,0,28,-77,3
100004,-24,3.0,0.0,0,0,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,3,0,0,0,1,0,0,0,0,4,-27,1
100005,-15,9.0,0.0,0,0,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,9,0,0,0,1,0,0,1,0,11,-25,1


In [116]:
# Add POCA Score generated by GRU
agg_poca = poca_recent.merge(right = poca_score.reset_index(), how = "left", on = "SK_ID_CURR")
agg_poca.head()

,SK_ID_CURR,poca_recent_MONTHS_BALANCE,poca_recent_CNT_INSTALMENT,poca_recent_CNT_INSTALMENT_FUTURE,poca_recent_NAME_CONTRACT_STATUS,poca_recent_SK_DPD,poca_recent_SK_DPD_DEF,poca_max_SK_DPD,poca_max_SK_DPD_DEF,poca_mean_SK_DPD,poca_mean_SK_DPD_DEF,poca_sum_SK_DPD_SCALE,poca_sum_SK_DPD_DEF_SCALE,poca_mean_SK_DPD_SCALE,poca_mean_SK_DPD_DEF_SCALE,poca_MONTH_LAST_DPD,poca_NAME_CONTRACT_STATUS_CNT_Active,poca_NAME_CONTRACT_STATUS_CNT_Amortized debt,poca_NAME_CONTRACT_STATUS_CNT_Approved,poca_NAME_CONTRACT_STATUS_CNT_Canceled,poca_NAME_CONTRACT_STATUS_CNT_Completed,poca_NAME_CONTRACT_STATUS_CNT_Demand,poca_NAME_CONTRACT_STATUS_CNT_Returned to the store,poca_NAME_CONTRACT_STATUS_CNT_Signed,poca_NAME_CONTRACT_STATUS_CNT_XNA,poca_MONTH_CNT,poca_MONTH_MAX,poca_count,pos_score
0,100001,-53,4.0,0.0,0,0,0,7,7,0.777778,0.777778,0.002552,0.002552,0.048169,0.048169,-95.0,7,0,0,0,2,0,0,0,0,9,-96,2,0.515204
1,100002,-1,24.0,6.0,1,0,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,19,0,0,0,0,0,0,0,0,19,-19,1,0.465684
2,100003,-18,7.0,0.0,0,0,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,26,0,0,0,2,0,0,0,0,28,-77,3,0.411413
3,100004,-24,3.0,0.0,0,0,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,3,0,0,0,1,0,0,0,0,4,-27,1,0.468323
4,100005,-15,9.0,0.0,0,0,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,9,0,0,0,1,0,0,1,0,11,-25,1,0.515069


#### 4.3.3 CARD DATA

In [117]:
card.head()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,AMT_PAYMENT_CURRENT,AMT_PAYMENT_TOTAL_CURRENT,AMT_RECEIVABLE_PRINCIPAL,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.970,135000,0.0,877.5,0.0,877.5,1700.325,1800.0,1800.0,0.000,0.000,0.000,0.0,1,0.0,1.0,35.0,Active,0,0
1,2582071,363914,-1,63975.555,45000,2250.0,2250.0,0.0,0.0,2250.000,2250.0,2250.0,60175.080,64875.555,64875.555,1.0,1,0.0,0.0,69.0,Active,0,0
2,1740877,371185,-7,31815.225,450000,0.0,0.0,0.0,0.0,2250.000,2250.0,2250.0,26926.425,31460.085,31460.085,0.0,0,0.0,0.0,30.0,Active,0,0
3,1389973,337855,-4,236572.110,225000,2250.0,2250.0,0.0,0.0,11795.760,11925.0,11925.0,224949.285,233048.970,233048.970,1.0,1,0.0,0.0,10.0,Active,0,0
4,1891521,126868,-1,453919.455,450000,0.0,11547.0,0.0,11547.0,22924.890,27000.0,27000.0,443044.395,453919.455,453919.455,0.0,1,0.0,1.0,101.0,Active,0,0


In [118]:
### Score file
train_cc_score = pd.read_csv('../data/prepared/cc_score_train.csv',index_col=0)
test_cc_score = pd.read_csv('../data/prepared/cc_score_test.csv',index_col=0)
card_score= pd.concat([train_cc_score, test_cc_score])
card_score.head()

,cc_score
SK_ID_CURR,
100006,0.376728
100011,0.482172
100021,0.359749
100023,0.392366
100036,0.417129


In [119]:
card_target1 = card.SK_ID_PREV.loc[card.SK_DPD_DEF > 0].unique()

sum_feats = [f_ for f_ in card.columns.values if ((f_.find('AMT') >= 0) | (f_.find('SK_DPD') >= 0) | (f_.find('CNT') >= 0) & (f_.find('CUM') == -1))]
print('sum_feats', sum_feats)

sum_card_mon = card.groupby(['SK_ID_CURR', 'MONTHS_BALANCE'])[sum_feats].sum()
sum_card_mon['CNT_ACCOUNT_W_MONTH'] = card.groupby(['SK_ID_CURR', 'MONTHS_BALANCE'])['SK_ID_PREV'].count()
sum_card_mon = sum_card_mon.reset_index()

# compute ratio after summing up account
sum_card_mon['AMT_BALANCE_CREDIT_RATIO'] = (sum_card_mon['AMT_BALANCE'] / (sum_card_mon['AMT_CREDIT_LIMIT_ACTUAL'] + 0.001)).clip(-100, 100)
sum_card_mon['AMT_CREDIT_USE_RATIO'] = (sum_card_mon['AMT_DRAWINGS_CURRENT'] / (sum_card_mon['AMT_CREDIT_LIMIT_ACTUAL'] + 0.001)).clip(-100, 100)
sum_card_mon['AMT_DRAWING_ATM_RATIO'] = sum_card_mon['AMT_DRAWINGS_ATM_CURRENT'] / (sum_card_mon['AMT_DRAWINGS_CURRENT'] + 0.001)
sum_card_mon['AMT_DRAWINGS_OTHER_RATIO'] = sum_card_mon['AMT_DRAWINGS_OTHER_CURRENT'] / (sum_card_mon['AMT_DRAWINGS_CURRENT'] + 0.001)
sum_card_mon['AMT_DRAWINGS_POS_RATIO'] = sum_card_mon['AMT_DRAWINGS_POS_CURRENT'] / (sum_card_mon['AMT_DRAWINGS_CURRENT'] + 0.001)
sum_card_mon['AMT_PAY_USE_RATIO'] = ((sum_card_mon['AMT_PAYMENT_TOTAL_CURRENT'] + 0.001) / (sum_card_mon['AMT_DRAWINGS_CURRENT'] + 0.001)).clip(-100, 100)
sum_card_mon['AMT_BALANCE_RECIVABLE_RATIO'] = sum_card_mon['AMT_BALANCE'] / (sum_card_mon['AMT_TOTAL_RECEIVABLE'] + 0.001)
sum_card_mon['AMT_DRAWING_BALANCE_RATIO'] = sum_card_mon['AMT_DRAWINGS_CURRENT'] / (sum_card_mon['AMT_BALANCE'] + 0.001)
sum_card_mon['AMT_RECEIVABLE_PRINCIPAL_DIFF'] = sum_card_mon['AMT_TOTAL_RECEIVABLE'] - sum_card_mon['AMT_RECEIVABLE_PRINCIPAL']
sum_card_mon['AMT_PAY_INST_DIFF'] = sum_card_mon['AMT_PAYMENT_CURRENT'] - sum_card_mon['AMT_INST_MIN_REGULARITY']

rejected_features = ['AMT_RECIVABLE', 'AMT_RECEIVABLE_PRINCIPAL',
                     'AMT_DRAWINGS_OTHER_CURRENT', 'AMT_DRAWINGS_POS_CURRENT']
for f_ in rejected_features:
    del sum_card_mon[f_]

sum_feats = [f_ for f_ in sum_card_mon.columns.values if ((f_.find('AMT') >= 0) | (f_.find('SK_DPD') >= 0) | (f_.find('CNT') >= 0) & (f_.find('CUM') == -1))]
print('updated sum_feats', sum_feats)

print('compute mean for different windows')
mean4_card_mon = sum_card_mon.loc[sum_card_mon.MONTHS_BALANCE >= -4].groupby('SK_ID_CURR').mean()
del mean4_card_mon['MONTHS_BALANCE']
mean4_card_mon.columns = ['mean4_' + f_ for f_ in mean4_card_mon.columns]

mean12_card_mon = sum_card_mon.loc[sum_card_mon.MONTHS_BALANCE >= -12].groupby('SK_ID_CURR').mean()
del mean12_card_mon['MONTHS_BALANCE']
mean12_card_mon.columns = ['mean12_' + f_ for f_ in mean12_card_mon.columns]

mean36_card_mon = sum_card_mon.loc[sum_card_mon.MONTHS_BALANCE >= -36].groupby('SK_ID_CURR').mean()
del mean36_card_mon['MONTHS_BALANCE']
mean36_card_mon.columns = ['mean36_' + f_ for f_ in mean36_card_mon.columns]

# sum_card_mon2 for scale features
print('compute scaled sum and mean')
sum_card_mon2 = sum_card_mon.copy(deep=True)
sum_card_mon2['YEAR_SCALE'] = (sum_card_mon2['MONTHS_BALANCE'] / 12.0).apply(np.exp)
for f_ in sum_feats:
    sum_card_mon2[f_] = sum_card_mon2[f_] * sum_card_mon2['YEAR_SCALE']

# scale sum
scale_sum_card_mon = sum_card_mon2.groupby('SK_ID_CURR').sum()
del scale_sum_card_mon['MONTHS_BALANCE'], scale_sum_card_mon['YEAR_SCALE']
scale_sum_card_mon.columns = ['scale_sum_' + f_ for f_ in scale_sum_card_mon.columns]

# scale mean
year_scale_sum = sum_card_mon2.groupby('SK_ID_CURR')['YEAR_SCALE'].sum()
scale_mean_card_mon = pd.DataFrame()
for f_ in scale_sum_card_mon.columns:
    scale_mean_card_mon[f_] = scale_sum_card_mon[f_] / year_scale_sum
scale_mean_card_mon.columns = ['scale_mean_' + f_ for f_ in scale_mean_card_mon.columns]

print('compute mean, var, max, min for all months')
# mean
del sum_card_mon['MONTHS_BALANCE']
mean_card_mon = sum_card_mon.groupby('SK_ID_CURR').mean()
mean_card_mon.columns = ['mean_' + f_ for f_ in mean_card_mon.columns]
# var
var_card_mon = sum_card_mon.groupby('SK_ID_CURR').var()
var_card_mon.columns = ['var_' + f_ for f_ in var_card_mon.columns]
# max
max_card_mon = sum_card_mon.groupby('SK_ID_CURR').max()
max_card_mon.columns = ['max_' + f_ for f_ in max_card_mon.columns]
# min
min_card_mon = sum_card_mon.groupby('SK_ID_CURR')[['AMT_TOTAL_RECEIVABLE', 'AMT_RECEIVABLE_PRINCIPAL_DIFF']].min()
min_card_mon.columns = ['min_' + f_ for f_ in min_card_mon.columns]

print('find last time with DPD')
# what is the last month with DPD
card_last_DPD = card[card.SK_DPD > 0].groupby(['SK_ID_CURR'])['MONTHS_BALANCE'].max()
card_last_DPD.rename('MONTH_LAST_DPD', inplace=True)

# what is the last month with 7 Days Past Due
card_last_DPD7 = card[card.SK_DPD_DEF > 7].groupby(['SK_ID_CURR'])['MONTHS_BALANCE'].max()
card_last_DPD7.rename('MONTH_LAST_DPD7', inplace=True)

# combine all
card_mon = mean4_card_mon.copy(deep=True)
card_mon = card_mon.merge(mean12_card_mon, how='outer', on='SK_ID_CURR')
card_mon = card_mon.merge(mean36_card_mon, how='outer', on='SK_ID_CURR')
card_mon = card_mon.merge(scale_sum_card_mon, how='outer', on='SK_ID_CURR')
card_mon = card_mon.merge(scale_mean_card_mon, how='outer', on='SK_ID_CURR')
card_mon = card_mon.merge(mean_card_mon, how='outer', on='SK_ID_CURR')
card_mon = card_mon.merge(var_card_mon, how='outer', on='SK_ID_CURR')
card_mon = card_mon.merge(max_card_mon, how='outer', on='SK_ID_CURR')
card_mon = card_mon.merge(min_card_mon, how='outer', on='SK_ID_CURR')
card_mon['MONTH_LAST_DPD'] = card_last_DPD
card_mon['MONTH_LAST_DPD7'] = card_last_DPD7
card_mon['MONTH_LAST_DPD'].loc[card_mon['MONTH_LAST_DPD'] == 0] = np.nan
card_mon['MONTH_LAST_DPD7'].loc[card_mon['MONTH_LAST_DPD7'] == 0] = np.nan

# most recent data
print('extract most recent data for each customer')
idx = card.groupby(['SK_ID_CURR'])['MONTHS_BALANCE'].idxmax()
recent = card[['SK_ID_CURR', 'MONTHS_BALANCE', 'CNT_INSTALMENT_MATURE_CUM',
               'NAME_CONTRACT_STATUS', 'SK_DPD', 'SK_DPD_DEF']].iloc[idx.values].copy(deep=True)

# most recent NAME_CONTRACT_STATUS for mean encoding
recent['NAME_CONTRACT_STATUS'], indexer = pd.factorize(recent['NAME_CONTRACT_STATUS'])
# meanenc_feats.append('card_NAME_CONTRACT_STATUS')
recent.set_index('SK_ID_CURR', inplace=True)

NAME_CONTRACT_STATUS_COUNT = pd.Series(card.groupby(['SK_ID_CURR'])['NAME_CONTRACT_STATUS'].value_counts(),
                                       name='NAME_CONTRACT_STATUS_COUNT')
NAME_CONTRACT_STATUS_COUNT = pd.pivot_table(NAME_CONTRACT_STATUS_COUNT.reset_index(),
                                            index='SK_ID_CURR', columns='NAME_CONTRACT_STATUS',
                                            values='NAME_CONTRACT_STATUS_COUNT', fill_value=0)

recent = recent.merge(NAME_CONTRACT_STATUS_COUNT, how='outer', on='SK_ID_CURR')
card_mon = card_mon.merge(recent, how='outer', on='SK_ID_CURR')

card['history_len'] = card.groupby('SK_ID_CURR')['MONTHS_BALANCE'].count()

#########
card_mon.fillna(0, inplace=True)
card_mon.columns = ['card_' + f_ for f_ in card_mon.columns]
card_mon = downcast_dtypes(card_mon)

del sum_card_mon, sum_card_mon2
del mean4_card_mon, mean12_card_mon, mean36_card_mon, recent
del scale_sum_card_mon, scale_mean_card_mon, mean_card_mon, var_card_mon, max_card_mon
#del card
gc.collect()
card_mon.head()

sum_feats ['AMT_BALANCE', 'AMT_CREDIT_LIMIT_ACTUAL', 'AMT_DRAWINGS_ATM_CURRENT', 'AMT_DRAWINGS_CURRENT', 'AMT_DRAWINGS_OTHER_CURRENT', 'AMT_DRAWINGS_POS_CURRENT', 'AMT_INST_MIN_REGULARITY', 'AMT_PAYMENT_CURRENT', 'AMT_PAYMENT_TOTAL_CURRENT', 'AMT_RECEIVABLE_PRINCIPAL', 'AMT_RECIVABLE', 'AMT_TOTAL_RECEIVABLE', 'CNT_DRAWINGS_ATM_CURRENT', 'CNT_DRAWINGS_CURRENT', 'CNT_DRAWINGS_OTHER_CURRENT', 'CNT_DRAWINGS_POS_CURRENT', 'SK_DPD', 'SK_DPD_DEF']
updated sum_feats ['AMT_BALANCE', 'AMT_CREDIT_LIMIT_ACTUAL', 'AMT_DRAWINGS_ATM_CURRENT', 'AMT_DRAWINGS_CURRENT', 'AMT_INST_MIN_REGULARITY', 'AMT_PAYMENT_CURRENT', 'AMT_PAYMENT_TOTAL_CURRENT', 'AMT_TOTAL_RECEIVABLE', 'CNT_DRAWINGS_ATM_CURRENT', 'CNT_DRAWINGS_CURRENT', 'CNT_DRAWINGS_OTHER_CURRENT', 'CNT_DRAWINGS_POS_CURRENT', 'SK_DPD', 'SK_DPD_DEF', 'CNT_ACCOUNT_W_MONTH', 'AMT_BALANCE_CREDIT_RATIO', 'AMT_CREDIT_USE_RATIO', 'AMT_DRAWING_ATM_RATIO', 'AMT_DRAWINGS_OTHER_RATIO', 'AMT_DRAWINGS_POS_RATIO', 'AMT_PAY_USE_RATIO', 'AMT_BALANCE_RECIVABLE_RATIO',

,card_mean4_AMT_BALANCE,card_mean4_AMT_CREDIT_LIMIT_ACTUAL,card_mean4_AMT_DRAWINGS_ATM_CURRENT,card_mean4_AMT_DRAWINGS_CURRENT,card_mean4_AMT_INST_MIN_REGULARITY,card_mean4_AMT_PAYMENT_CURRENT,card_mean4_AMT_PAYMENT_TOTAL_CURRENT,card_mean4_AMT_TOTAL_RECEIVABLE,card_mean4_CNT_DRAWINGS_ATM_CURRENT,card_mean4_CNT_DRAWINGS_CURRENT,card_mean4_CNT_DRAWINGS_OTHER_CURRENT,card_mean4_CNT_DRAWINGS_POS_CURRENT,card_mean4_SK_DPD,card_mean4_SK_DPD_DEF,card_mean4_CNT_ACCOUNT_W_MONTH,card_mean4_AMT_BALANCE_CREDIT_RATIO,card_mean4_AMT_CREDIT_USE_RATIO,card_mean4_AMT_DRAWING_ATM_RATIO,card_mean4_AMT_DRAWINGS_OTHER_RATIO,card_mean4_AMT_DRAWINGS_POS_RATIO,card_mean4_AMT_PAY_USE_RATIO,card_mean4_AMT_BALANCE_RECIVABLE_RATIO,card_mean4_AMT_DRAWING_BALANCE_RATIO,card_mean4_AMT_RECEIVABLE_PRINCIPAL_DIFF,card_mean4_AMT_PAY_INST_DIFF,card_mean12_AMT_BALANCE,card_mean12_AMT_CREDIT_LIMIT_ACTUAL,card_mean12_AMT_DRAWINGS_ATM_CURRENT,card_mean12_AMT_DRAWINGS_CURRENT,card_mean12_AMT_INST_MIN_REGULARITY,card_mean12_AMT_PAYMENT_CURRENT,card_mean12_AMT_PAYMENT_TOTAL_CURRENT,card_mean12_AMT_TOTAL_RECEIVABLE,card_mean12_CNT_DRAWINGS_ATM_CURRENT,card_mean12_CNT_DRAWINGS_CURRENT,card_mean12_CNT_DRAWINGS_OTHER_CURRENT,card_mean12_CNT_DRAWINGS_POS_CURRENT,card_mean12_SK_DPD,card_mean12_SK_DPD_DEF,card_mean12_CNT_ACCOUNT_W_MONTH,card_mean12_AMT_BALANCE_CREDIT_RATIO,card_mean12_AMT_CREDIT_USE_RATIO,card_mean12_AMT_DRAWING_ATM_RATIO,card_mean12_AMT_DRAWINGS_OTHER_RATIO,card_mean12_AMT_DRAWINGS_POS_RATIO,card_mean12_AMT_PAY_USE_RATIO,card_mean12_AMT_BALANCE_RECIVABLE_RATIO,card_mean12_AMT_DRAWING_BALANCE_RATIO,card_mean12_AMT_RECEIVABLE_PRINCIPAL_DIFF,card_mean12_AMT_PAY_INST_DIFF,card_mean36_AMT_BALANCE,card_mean36_AMT_CREDIT_LIMIT_ACTUAL,card_mean36_AMT_DRAWINGS_ATM_CURRENT,card_mean36_AMT_DRAWINGS_CURRENT,card_mean36_AMT_INST_MIN_REGULARITY,card_mean36_AMT_PAYMENT_CURRENT,card_mean36_AMT_PAYMENT_TOTAL_CURRENT,card_mean36_AMT_TOTAL_RECEIVABLE,card_mean36_CNT_DRAWINGS_ATM_CURRENT,card_mean36_CNT_DRAWINGS_CURRENT,card_mean36_CNT_DRAWINGS_OTHER_CURRENT,card_mean36_CNT_DRAWINGS_POS_CURRENT,card_mean36_SK_DPD,card_mean36_SK_DPD_DEF,card_mean36_CNT_ACCOUNT_W_MONTH,card_mean36_AMT_BALANCE_CREDIT_RATIO,card_mean36_AMT_CREDIT_USE_RATIO,card_mean36_AMT_DRAWING_ATM_RATIO,card_mean36_AMT_DRAWINGS_OTHER_RATIO,card_mean36_AMT_DRAWINGS_POS_RATIO,card_mean36_AMT_PAY_USE_RATIO,card_mean36_AMT_BALANCE_RECIVABLE_RATIO,card_mean36_AMT_DRAWING_BALANCE_RATIO,card_mean36_AMT_RECEIVABLE_PRINCIPAL_DIFF,card_mean36_AMT_PAY_INST_DIFF,card_scale_sum_AMT_BALANCE,card_scale_sum_AMT_CREDIT_LIMIT_ACTUAL,card_scale_sum_AMT_DRAWINGS_ATM_CURRENT,card_scale_sum_AMT_DRAWINGS_CURRENT,card_scale_sum_AMT_INST_MIN_REGULARITY,card_scale_sum_AMT_PAYMENT_CURRENT,card_scale_sum_AMT_PAYMENT_TOTAL_CURRENT,card_scale_sum_AMT_TOTAL_RECEIVABLE,card_scale_sum_CNT_DRAWINGS_ATM_CURRENT,card_scale_sum_CNT_DRAWINGS_CURRENT,card_scale_sum_CNT_DRAWINGS_OTHER_CURRENT,card_scale_sum_CNT_DRAWINGS_POS_CURRENT,card_scale_sum_SK_DPD,card_scale_sum_SK_DPD_DEF,card_scale_sum_CNT_ACCOUNT_W_MONTH,card_scale_sum_AMT_BALANCE_CREDIT_RATIO,card_scale_sum_AMT_CREDIT_USE_RATIO,card_scale_sum_AMT_DRAWING_ATM_RATIO,card_scale_sum_AMT_DRAWINGS_OTHER_RATIO,card_scale_sum_AMT_DRAWINGS_POS_RATIO,card_scale_sum_AMT_PAY_USE_RATIO,card_scale_sum_AMT_BALANCE_RECIVABLE_RATIO,card_scale_sum_AMT_DRAWING_BALANCE_RATIO,card_scale_sum_AMT_RECEIVABLE_PRINCIPAL_DIFF,card_scale_sum_AMT_PAY_INST_DIFF,card_scale_mean_scale_sum_AMT_BALANCE,card_scale_mean_scale_sum_AMT_CREDIT_LIMIT_ACTUAL,card_scale_mean_scale_sum_AMT_DRAWINGS_ATM_CURRENT,card_scale_mean_scale_sum_AMT_DRAWINGS_CURRENT,card_scale_mean_scale_sum_AMT_INST_MIN_REGULARITY,card_scale_mean_scale_sum_AMT_PAYMENT_CURRENT,card_scale_mean_scale_sum_AMT_PAYMENT_TOTAL_CURRENT,card_scale_mean_scale_sum_AMT_TOTAL_RECEIVABLE,card_scale_mean_scale_sum_CNT_DRAWINGS_ATM_CURRENT,card_scale_mean_scale_sum_CNT_DRAWINGS_CURRENT,card_scale_mean_scale_sum_CNT_DRAWINGS_OTHER_CURRENT,card_scale_mean_scale_sum_CNT_DRAWING

In [120]:
# Add CARD Score generated by GRU
agg_card = card_mon.merge(right = card_score.reset_index(), how = "left", on = "SK_ID_CURR")
agg_card.head()

,SK_ID_CURR,card_mean4_AMT_BALANCE,card_mean4_AMT_CREDIT_LIMIT_ACTUAL,card_mean4_AMT_DRAWINGS_ATM_CURRENT,card_mean4_AMT_DRAWINGS_CURRENT,card_mean4_AMT_INST_MIN_REGULARITY,card_mean4_AMT_PAYMENT_CURRENT,card_mean4_AMT_PAYMENT_TOTAL_CURRENT,card_mean4_AMT_TOTAL_RECEIVABLE,card_mean4_CNT_DRAWINGS_ATM_CURRENT,card_mean4_CNT_DRAWINGS_CURRENT,card_mean4_CNT_DRAWINGS_OTHER_CURRENT,card_mean4_CNT_DRAWINGS_POS_CURRENT,card_mean4_SK_DPD,card_mean4_SK_DPD_DEF,card_mean4_CNT_ACCOUNT_W_MONTH,card_mean4_AMT_BALANCE_CREDIT_RATIO,card_mean4_AMT_CREDIT_USE_RATIO,card_mean4_AMT_DRAWING_ATM_RATIO,card_mean4_AMT_DRAWINGS_OTHER_RATIO,card_mean4_AMT_DRAWINGS_POS_RATIO,card_mean4_AMT_PAY_USE_RATIO,card_mean4_AMT_BALANCE_RECIVABLE_RATIO,card_mean4_AMT_DRAWING_BALANCE_RATIO,card_mean4_AMT_RECEIVABLE_PRINCIPAL_DIFF,card_mean4_AMT_PAY_INST_DIFF,card_mean12_AMT_BALANCE,card_mean12_AMT_CREDIT_LIMIT_ACTUAL,card_mean12_AMT_DRAWINGS_ATM_CURRENT,card_mean12_AMT_DRAWINGS_CURRENT,card_mean12_AMT_INST_MIN_REGULARITY,card_mean12_AMT_PAYMENT_CURRENT,card_mean12_AMT_PAYMENT_TOTAL_CURRENT,card_mean12_AMT_TOTAL_RECEIVABLE,card_mean12_CNT_DRAWINGS_ATM_CURRENT,card_mean12_CNT_DRAWINGS_CURRENT,card_mean12_CNT_DRAWINGS_OTHER_CURRENT,card_mean12_CNT_DRAWINGS_POS_CURRENT,card_mean12_SK_DPD,card_mean12_SK_DPD_DEF,card_mean12_CNT_ACCOUNT_W_MONTH,card_mean12_AMT_BALANCE_CREDIT_RATIO,card_mean12_AMT_CREDIT_USE_RATIO,card_mean12_AMT_DRAWING_ATM_RATIO,card_mean12_AMT_DRAWINGS_OTHER_RATIO,card_mean12_AMT_DRAWINGS_POS_RATIO,card_mean12_AMT_PAY_USE_RATIO,card_mean12_AMT_BALANCE_RECIVABLE_RATIO,card_mean12_AMT_DRAWING_BALANCE_RATIO,card_mean12_AMT_RECEIVABLE_PRINCIPAL_DIFF,card_mean12_AMT_PAY_INST_DIFF,card_mean36_AMT_BALANCE,card_mean36_AMT_CREDIT_LIMIT_ACTUAL,card_mean36_AMT_DRAWINGS_ATM_CURRENT,card_mean36_AMT_DRAWINGS_CURRENT,card_mean36_AMT_INST_MIN_REGULARITY,card_mean36_AMT_PAYMENT_CURRENT,card_mean36_AMT_PAYMENT_TOTAL_CURRENT,card_mean36_AMT_TOTAL_RECEIVABLE,card_mean36_CNT_DRAWINGS_ATM_CURRENT,card_mean36_CNT_DRAWINGS_CURRENT,card_mean36_CNT_DRAWINGS_OTHER_CURRENT,card_mean36_CNT_DRAWINGS_POS_CURRENT,card_mean36_SK_DPD,card_mean36_SK_DPD_DEF,card_mean36_CNT_ACCOUNT_W_MONTH,card_mean36_AMT_BALANCE_CREDIT_RATIO,card_mean36_AMT_CREDIT_USE_RATIO,card_mean36_AMT_DRAWING_ATM_RATIO,card_mean36_AMT_DRAWINGS_OTHER_RATIO,card_mean36_AMT_DRAWINGS_POS_RATIO,card_mean36_AMT_PAY_USE_RATIO,card_mean36_AMT_BALANCE_RECIVABLE_RATIO,card_mean36_AMT_DRAWING_BALANCE_RATIO,card_mean36_AMT_RECEIVABLE_PRINCIPAL_DIFF,card_mean36_AMT_PAY_INST_DIFF,card_scale_sum_AMT_BALANCE,card_scale_sum_AMT_CREDIT_LIMIT_ACTUAL,card_scale_sum_AMT_DRAWINGS_ATM_CURRENT,card_scale_sum_AMT_DRAWINGS_CURRENT,card_scale_sum_AMT_INST_MIN_REGULARITY,card_scale_sum_AMT_PAYMENT_CURRENT,card_scale_sum_AMT_PAYMENT_TOTAL_CURRENT,card_scale_sum_AMT_TOTAL_RECEIVABLE,card_scale_sum_CNT_DRAWINGS_ATM_CURRENT,card_scale_sum_CNT_DRAWINGS_CURRENT,card_scale_sum_CNT_DRAWINGS_OTHER_CURRENT,card_scale_sum_CNT_DRAWINGS_POS_CURRENT,card_scale_sum_SK_DPD,card_scale_sum_SK_DPD_DEF,card_scale_sum_CNT_ACCOUNT_W_MONTH,card_scale_sum_AMT_BALANCE_CREDIT_RATIO,card_scale_sum_AMT_CREDIT_USE_RATIO,card_scale_sum_AMT_DRAWING_ATM_RATIO,card_scale_sum_AMT_DRAWINGS_OTHER_RATIO,card_scale_sum_AMT_DRAWINGS_POS_RATIO,card_scale_sum_AMT_PAY_USE_RATIO,card_scale_sum_AMT_BALANCE_RECIVABLE_RATIO,card_scale_sum_AMT_DRAWING_BALANCE_RATIO,card_scale_sum_AMT_RECEIVABLE_PRINCIPAL_DIFF,card_scale_sum_AMT_PAY_INST_DIFF,card_scale_mean_scale_sum_AMT_BALANCE,card_scale_mean_scale_sum_AMT_CREDIT_LIMIT_ACTUAL,card_scale_mean_scale_sum_AMT_DRAWINGS_ATM_CURRENT,card_scale_mean_scale_sum_AMT_DRAWINGS_CURRENT,card_scale_mean_scale_sum_AMT_INST_MIN_REGULARITY,card_scale_mean_scale_sum_AMT_PAYMENT_CURRENT,card_scale_mean_scale_sum_AMT_PAYMENT_TOTAL_CURRENT,card_scale_mean_scale_sum_AMT_TOTAL_RECEIVABLE,card_scale_mean_scale_sum_CNT_DRAWINGS_ATM_CURRENT,card_scale_mean_scale_sum_CNT_DRAWINGS_CURRENT,card_scale_mean_scale_sum_CNT_DRAWINGS_OTHER_CURRENT,card_scale_mean_scale_sum_

In [121]:
# clear memory
del card

#### 4.3.4. PREV Data

In [122]:
# check card data
prev.head()

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,FLAG_LAST_APPL_PER_CONTRACT,NFLAG_LAST_APPL_IN_DAY,RATE_DOWN_PAYMENT,RATE_INTEREST_PRIMARY,RATE_INTEREST_PRIVILEGED,NAME_CASH_LOAN_PURPOSE,NAME_CONTRACT_STATUS,DAYS_DECISION,NAME_PAYMENT_TYPE,CODE_REJECT_REASON,NAME_TYPE_SUITE,NAME_CLIENT_TYPE,NAME_GOODS_CATEGORY,NAME_PORTFOLIO,NAME_PRODUCT_TYPE,CHANNEL_TYPE,SELLERPLACE_AREA,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,Y,1,0.0,0.182832,0.867336,XAP,Approved,-73,Cash through the bank,XAP,NaN,Repeater,Mobile,POS,XNA,Country-wide,35,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,Y,1,NaN,NaN,NaN,XNA,Approved,-164,XNA,XAP,Unaccompanied,Repeater,XNA,Cash,x-sell,Contact center,-1,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,Y,1,NaN,NaN,NaN,XNA,Approved,-301,Cash through the bank,XAP,"Spouse, partner",Repeater,XNA,Cash,x-sell,Credit and cash offices,-1,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,Y,1,NaN,NaN,NaN,XNA,Approved,-512,Cash through the bank,XAP,NaN,Repeater,XNA,Cash,x-sell,Credit and cash offices,-1,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,Y,1,NaN,NaN,NaN,Repairs,Refused,-781,Cash through the bank,HC,NaN,Repeater,XNA,Cash,walk-in,Credit and cash offices,-1,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
### FEATURE ENGINEERING

# amount ratios
prev["AMT_GIVEN_RATIO_1"]  = prev["AMT_CREDIT"] / prev["AMT_APPLICATION"]
prev["AMT_GIVEN_RATIO_2"]  = prev["AMT_GOODS_PRICE"] / prev["AMT_APPLICATION"]
prev["DOWN_PAYMENT_RATIO"] = prev["AMT_DOWN_PAYMENT"] / prev["AMT_APPLICATION"]

# number of applications 
cnt_prev = prev[["SK_ID_CURR", "SK_ID_PREV"]].groupby(["SK_ID_CURR"], as_index = False).count()
cnt_prev.columns = ["SK_ID_CURR", "CNT_PREV_APPLICATIONS"]
prev = prev.merge(cnt_prev, how = "left", on = "SK_ID_CURR")

# number of contracts
cnt_prev = prev[["SK_ID_CURR", "FLAG_LAST_APPL_PER_CONTRACT"]]
cnt_prev.columns = ["SK_ID_CURR", "CNT_PREV_CONTRACTS"]
cnt_prev = cnt_prev[cnt_prev["CNT_PREV_CONTRACTS"] == "Y"]
cnt_prev = cnt_prev[["SK_ID_CURR", "CNT_PREV_CONTRACTS"]].groupby(["SK_ID_CURR"], as_index = False).count()
prev = prev.merge(cnt_prev, how = "left", on = "SK_ID_CURR")

# number ratio
prev["APPL_PER_CONTRACT_RATIO"] = prev["CNT_PREV_APPLICATIONS"] / prev["CNT_PREV_CONTRACTS"]

# loan decision ratios
prev = compute_accept_reject_ratio(prev, lags = [1, 3, 5])

# day differences
prev["DAYS_DUE_DIFF_1"] = prev["DAYS_LAST_DUE_1ST_VERSION"] - prev["DAYS_FIRST_DUE"]
prev["DAYS_DUE_DIFF_2"] = prev["DAYS_LAST_DUE"] - prev["DAYS_FIRST_DUE"]
prev["DAYS_TERMINATION_DIFF_1"] = prev["DAYS_TERMINATION"] - prev["DAYS_FIRST_DRAWING"]
prev["DAYS_TERMINATION_DIFF_2"] = prev["DAYS_TERMINATION"] - prev["DAYS_FIRST_DUE"]
prev["DAYS_TERMINATION_DIFF_3"] = prev["DAYS_TERMINATION"] - prev["DAYS_LAST_DUE"]

# application dates
prev["DAY_APPR_PROCESS_START"] = "Working day"
prev["DAY_APPR_PROCESS_START"][(prev["WEEKDAY_APPR_PROCESS_START"] == "SATURDAY") |
                               (prev["WEEKDAY_APPR_PROCESS_START"] == "SUNDAY")] = "Weekend"


##### FEATURE REMOVAL
drops = ["NAME_CLIENT_TYPE", "SK_ID_PREV"]
prev = prev.drop(columns = drops)

In [124]:
# dummy encodnig for factors
prev = pd.get_dummies(prev, drop_first = True)

In [125]:
# count missings
nas = count_missings(prev)
nas.head()

,Total,Percent
RATE_INTEREST_PRIMARY,1664263,99.643698
RATE_INTEREST_PRIVILEGED,1664263,99.643698
DOWN_PAYMENT_RATIO,895869,53.637977
AMT_DOWN_PAYMENT,895844,53.636480
RATE_DOWN_PAYMENT,895844,53.636480


In [126]:
### AGGREGATIONS

# aggregate data
agg_prev = aggregate_data(prev, id_var = "SK_ID_CURR", label = "prev")

# clean up
omits = ["APPROVE_RATIO_1", "APPROVE_RATIO_3", "APPROVE_RATIO_5",  
         "REJECT_RATIO_1", "REJECT_RATIO_3",  "REJECT_RATIO_5", 
         "FLAG_LAST_APPL_PER_CONTRACT_Y", "CNT_PREV_CONTRACTS", "CNT_PREV_APPLICATIONS", 
         "APPL_PER_CONTRACT_RATIO"]
for var in omits:
    del agg_prev["prev_" + str(var) + "_std"]
    del agg_prev["prev_" + str(var) + "_min"]
    del agg_prev["prev_" + str(var) + "_max"]

- Preparing the dataset...


- Extracted 0 factors and 161 numerics...
- Aggregating numeric features...
- Final dimensions: (338857, 644)


In [127]:
# count missings
nas = count_missings(agg_prev)
nas.head()

,Total,Percent
prev_RATE_INTEREST_PRIVILEGED_std,338639,99.935666
prev_RATE_INTEREST_PRIMARY_std,338639,99.935666
prev_RATE_INTEREST_PRIMARY_mean,333136,98.311677
prev_RATE_INTEREST_PRIVILEGED_mean,333136,98.311677
prev_RATE_INTEREST_PRIVILEGED_min,333136,98.311677


In [128]:
# check data
agg_prev.head()

,prev_DAYS_LAST_DUE_mean,prev_DAYS_LAST_DUE_std,prev_DAYS_LAST_DUE_min,prev_DAYS_LAST_DUE_max,prev_NAME_CONTRACT_STATUS_Refused_mean,prev_NAME_CONTRACT_STATUS_Refused_std,prev_NAME_CONTRACT_STATUS_Refused_min,prev_NAME_CONTRACT_STATUS_Refused_max,prev_NAME_TYPE_SUITE_Group of people_mean,prev_NAME_TYPE_SUITE_Group of people_std,prev_NAME_TYPE_SUITE_Group of people_min,prev_NAME_TYPE_SUITE_Group of people_max,prev_NAME_CASH_LOAN_PURPOSE_Wedding / gift / holiday_mean,prev_NAME_CASH_LOAN_PURPOSE_Wedding / gift / holiday_std,prev_NAME_CASH_LOAN_PURPOSE_Wedding / gift / holiday_min,prev_NAME_CASH_LOAN_PURPOSE_Wedding / gift / holiday_max,prev_SELLERPLACE_AREA_mean,prev_SELLERPLACE_AREA_std,prev_SELLERPLACE_AREA_min,prev_SELLERPLACE_AREA_max,prev_AMT_DOWN_PAYMENT_mean,prev_AMT_DOWN_PAYMENT_std,prev_AMT_DOWN_PAYMENT_min,prev_AMT_DOWN_PAYMENT_max,prev_AMT_GOODS_PRICE_mean,prev_AMT_GOODS_PRICE_std,prev_AMT_GOODS_PRICE_min,prev_AMT_GOODS_PRICE_max,prev_NAME_GOODS_CATEGORY_Medical Supplies_mean,prev_NAME_GOODS_CATEGORY_Medical Supplies_std,prev_NAME_GOODS_CATEGORY_Medical Supplies_min,prev_NAME_GOODS_CATEGORY_Medical Supplies_max,prev_NAME_GOODS_CATEGORY_Fitness_mean,prev_NAME_GOODS_CATEGORY_Fitness_std,prev_NAME_GOODS_CATEGORY_Fitness_min,prev_NAME_GOODS_CATEGORY_Fitness_max,prev_AMT_GIVEN_RATIO_1_mean,prev_AMT_GIVEN_RATIO_1_std,prev_AMT_GIVEN_RATIO_1_min,prev_AMT_GIVEN_RATIO_1_max,prev_PRODUCT_COMBINATION_Cash X-Sell: low_mean,prev_PRODUCT_COMBINATION_Cash X-Sell: low_std,prev_PRODUCT_COMBINATION_Cash X-Sell: low_min,prev_PRODUCT_COMBINATION_Cash X-Sell: low_max,prev_DOWN_PAYMENT_RATIO_mean,prev_DOWN_PAYMENT_RATIO_std,prev_DOWN_PAYMENT_RATIO_min,prev_DOWN_PAYMENT_RATIO_max,prev_NAME_SELLER_INDUSTRY_Industry_mean,prev_NAME_SELLER_INDUSTRY_Industry_std,prev_NAME_SELLER_INDUSTRY_Industry_min,prev_NAME_SELLER_INDUSTRY_Industry_max,prev_NAME_GOODS_CATEGORY_Consumer Electronics_mean,prev_NAME_GOODS_CATEGORY_Consumer Electronics_std,prev_NAME_GOODS_CATEGORY_Consumer Electronics_min,prev_NAME_GOODS_CATEGORY_Consumer Electronics_max,prev_NAME_CASH_LOAN_PURPOSE_Urgent needs_mean,prev_NAME_CASH_LOAN_PURPOSE_Urgent needs_std,prev_NAME_CASH_LOAN_PURPOSE_Urgent needs_min,prev_NAME_CASH_LOAN_PURPOSE_Urgent needs_max,prev_NAME_PRODUCT_TYPE_x-sell_mean,prev_NAME_PRODUCT_TYPE_x-sell_std,prev_NAME_PRODUCT_TYPE_x-sell_min,prev_NAME_PRODUCT_TYPE_x-sell_max,prev_AMT_APPLICATION_mean,prev_AMT_APPLICATION_std,prev_AMT_APPLICATION_min,prev_AMT_APPLICATION_max,prev_NAME_GOODS_CATEGORY_Auto Accessories_mean,prev_NAME_GOODS_CATEGORY_Auto Accessories_std,prev_NAME_GOODS_CATEGORY_Auto Accessories_min,prev_NAME_GOODS_CATEGORY_Auto Accessories_max,prev_NAME_GOODS_CATEGORY_Weapon_mean,prev_NAME_GOODS_CATEGORY_Weapon_std,prev_NAME_GOODS_CATEGORY_Weapon_min,prev_NAME_GOODS_CATEGORY_Weapon_max,prev_NAME_GOODS_CATEGORY_Animals_mean,prev_NAME_GOODS_CATEGORY_Animals_std,prev_NAME_GOODS_CATEGORY_Animals_min,prev_NAME_GOODS_CATEGORY_Animals_max,prev_DAYS_LAST_DUE_1ST_VERSION_mean,prev_DAYS_LAST_DUE_1ST_VERSION_std,prev_DAYS_LAST_DUE_1ST_VERSION_min,prev_DAYS_LAST_DUE_1ST_VERSION_max,prev_DAYS_TERMINATION_DIFF_1_mean,prev_DAYS_TERMINATION_DIFF_1_std,prev_DAYS_TERMINATION_DIFF_1_min,prev_DAYS_TERMINATION_DIFF_1_max,prev_WEEKDAY_APPR_PROCESS_START_THURSDAY_mean,prev_WEEKDAY_APPR_PROCESS_START_THURSDAY_std,prev_WEEKDAY_APPR_PROCESS_START_THURSDAY_min,prev_WEEKDAY_APPR_PROCESS_START_THURSDAY_max,prev_NAME_GOODS_CATEGORY_Direct Sales_mean,prev_NAME_GOODS_CATEGORY_Direct Sales_std,prev_NAME_GOODS_CATEGORY_Direct Sales_min,prev_NAME_GOODS_CATEGORY_Direct Sales_max,prev_NAME_GOODS_CATEGORY_Computers_mean,prev_NAME_GOODS_CATEGORY_Computers_std,prev_NAME_GOODS_CATEGORY_Computers_min,prev_NAME_GOODS_CATEGORY_Computers_max,prev_WEEKDAY_APPR_PROCESS_START_MONDAY_mean,prev_WEEKDAY_APPR_PROCESS_START_MONDAY_std,prev_WEEKDAY_APPR_PROCESS_START_MONDAY_min,prev_WEEKDAY_APPR_PROCESS_START_MONDAY_max,prev_NAME_SELLER_INDUSTRY_Consumer electronics_mean,prev_NAME_SELLER_INDUSTRY_Consumer electronic

In [129]:
# clear memory
del prev

## 5. Data Export

In [130]:
# merge data
print(appl.shape)
appl = appl.merge(right = agg_buro.reset_index(), how = "left", on = "SK_ID_CURR")
print(appl.shape)
#del agg_buro
appl = appl.merge(right = agg_prev.reset_index(), how = "left", on = "SK_ID_CURR")
print(appl.shape)
#del agg_prev
appl = appl.merge(right = agg_inst.reset_index(), how = "left", on = "SK_ID_CURR")
print(appl.shape)
#del agg_inst
appl = appl.merge(right = agg_poca.reset_index(), how = "left", on = "SK_ID_CURR")
print(appl.shape)
#del agg_poca
appl = appl.merge(right = agg_card.reset_index(), how = "left", on = "SK_ID_CURR")
print(appl.shape)
#del agg_card

(356255, 109)
(356255, 341)
(356255, 955)
(356255, 984)
(356255, 1013)
(356255, 1231)


In [131]:
##### CROSS-TABLE FEATURE ENGINEERING

# credit ratios
appl["mix_AMT_PREV_ANNUITY_RATIO"]     = appl["app_AMT_ANNUITY"] / appl["prev_AMT_ANNUITY_mean"]
appl["mix_AMT_PREV_CREDIT_RATIO"]      = appl["app_AMT_CREDIT"] / appl["prev_AMT_CREDIT_mean"]
appl["mix_AMT_PREV_GOODS_PRICE_RATIO"] = appl["app_AMT_GOODS_PRICE"] / appl["prev_AMT_GOODS_PRICE_mean"]
appl["mix_AMT_BURO_ANNUITY_RATIO"]     = appl["app_AMT_ANNUITY"] / appl["buro_AMT_ANNUITY_mean"]
appl["mix_AMT_BURO_CREDIT_RATIO"]      = appl["app_AMT_CREDIT"] / appl["buro_AMT_CREDIT_SUM_mean"]

In [132]:
# dummy encodnig for factors
appl = pd.get_dummies(appl, drop_first = True)

In [133]:
# label encoder for factors
#data_factors = [f for f in appl.columns if appl[f].dtype == "object"]
#for var in data_factors:
#    appl[var], _ = pd.factorize(appl[var])

In [134]:
appl.head()

,SK_ID_CURR,app_CNT_CHILDREN,app_AMT_INCOME_TOTAL,app_AMT_CREDIT,app_AMT_ANNUITY,app_AMT_GOODS_PRICE,app_REGION_POPULATION_RELATIVE,app_DAYS_BIRTH,app_DAYS_EMPLOYED,app_DAYS_REGISTRATION,app_DAYS_ID_PUBLISH,app_OWN_CAR_AGE,app_FLAG_MOBIL,app_FLAG_EMP_PHONE,app_FLAG_WORK_PHONE,app_FLAG_CONT_MOBILE,app_FLAG_PHONE,app_FLAG_EMAIL,app_CNT_FAM_MEMBERS,app_REGION_RATING_CLIENT,app_REGION_RATING_CLIENT_W_CITY,app_HOUR_APPR_PROCESS_START,app_REG_REGION_NOT_LIVE_REGION,app_REG_REGION_NOT_WORK_REGION,app_LIVE_REGION_NOT_WORK_REGION,app_REG_CITY_NOT_LIVE_CITY,app_REG_CITY_NOT_WORK_CITY,app_LIVE_CITY_NOT_WORK_CITY,app_EXT_SOURCE_1,app_EXT_SOURCE_2,app_EXT_SOURCE_3,app_APARTMENTS_AVG,app_BASEMENTAREA_AVG,app_YEARS_BEGINEXPLUATATION_AVG,app_YEARS_BUILD_AVG,app_COMMONAREA_AVG,app_ELEVATORS_AVG,app_ENTRANCES_AVG,app_FLOORSMAX_AVG,app_FLOORSMIN_AVG,app_LANDAREA_AVG,app_LIVINGAPARTMENTS_AVG,app_LIVINGAREA_AVG,app_NONLIVINGAPARTMENTS_AVG,app_NONLIVINGAREA_AVG,app_YEARS_BUILD_MODE,app_OBS_30_CNT_SOCIAL_CIRCLE,app_DEF_30_CNT_SOCIAL_CIRCLE,app_OBS_60_CNT_SOCIAL_CIRCLE,app_DEF_60_CNT_SOCIAL_CIRCLE,app_DAYS_LAST_PHONE_CHANGE,app_FLAG_DOCUMENT_2,app_FLAG_DOCUMENT_3,app_FLAG_DOCUMENT_4,app_FLAG_DOCUMENT_5,app_FLAG_DOCUMENT_6,app_FLAG_DOCUMENT_7,app_FLAG_DOCUMENT_8,app_FLAG_DOCUMENT_9,app_FLAG_DOCUMENT_10,app_FLAG_DOCUMENT_11,app_FLAG_DOCUMENT_12,app_FLAG_DOCUMENT_13,app_FLAG_DOCUMENT_14,app_FLAG_DOCUMENT_15,app_FLAG_DOCUMENT_16,app_FLAG_DOCUMENT_17,app_FLAG_DOCUMENT_18,app_FLAG_DOCUMENT_19,app_FLAG_DOCUMENT_20,app_FLAG_DOCUMENT_21,app_AMT_REQ_CREDIT_BUREAU_HOUR,app_AMT_REQ_CREDIT_BUREAU_DAY,app_AMT_REQ_CREDIT_BUREAU_WEEK,app_AMT_REQ_CREDIT_BUREAU_MON,app_AMT_REQ_CREDIT_BUREAU_QRT,app_AMT_REQ_CREDIT_BUREAU_YEAR,app_CREDIT_BY_INCOME,app_ANNUITY_BY_INCOME,app_GOODS_PRICE_BY_INCOME,app_INCOME_PER_PERSON,app_PERCENT_WORKED,app_CNT_ADULTS,app_CHILDREN_RATIO,app_ANNUITY LENGTH,app_EXT_SOURCE_MEAN,app_NUM_EXT_SOURCES,app_NUM_DOCUMENTS,app_OWN_CAR_AGE_RATIO,app_DAYS_ID_PUBLISHED_RATIO,app_DAYS_REGISTRATION_RATIO,app_DAYS_LAST_PHONE_CHANGE_RATIO,index_x,buro_CNT_BURO_CLOSED_mean,buro_CNT_BURO_CLOSED_std,buro_CNT_BURO_CLOSED_min,buro_CNT_BURO_CLOSED_max,buro_CREDIT_TYPE_Consumer credit_mean,buro_CREDIT_TYPE_Consumer credit_std,buro_CREDIT_TYPE_Consumer credit_min,buro_CREDIT_TYPE_Consumer credit_max,buro_STATUS_1_mean,buro_STATUS_1_std,buro_STATUS_1_min,buro_STATUS_1_max,buro_CREDIT_CURRENCY_currency 4_mean,buro_CREDIT_CURRENCY_currency 4_std,buro_CREDIT_CURRENCY_currency 4_min,buro_CREDIT_CURRENCY_currency 4_max,buro_CREDIT_TYPE_Real estate loan_mean,buro_CREDIT_TYPE_Real estate loan_std,buro_CREDIT_TYPE_Real estate loan_min,buro_CREDIT_TYPE_Real estate loan_max,buro_AMT_CREDIT_SUM_mean,buro_AMT_CREDIT_SUM_std,buro_AMT_CREDIT_SUM_min,buro_AMT_CREDIT_SUM_max,buro_CREDIT_TYPE_Credit card_mean,buro_CREDIT_TYPE_Credit card_std,buro_CREDIT_TYPE_Credit card_min,buro_CREDIT_TYPE_Credit card_max,buro_AMT_MAX_OVERDUE_RATIO_1_mean,buro_AMT_MAX_OVERDUE_RATIO_1_std,buro_AMT_MAX_OVERDUE_RATIO_1_min,buro_AMT_MAX_OVERDUE_RATIO_1_max,buro_CREDIT_TYPE_Interbank credit_mean,buro_CREDIT_TYPE_Interbank credit_std,buro_CREDIT_TYPE_Interbank credit_min,buro_CREDIT_TYPE_Interbank credit_max,buro_DAYS_CREDIT_mean,buro_DAYS_CREDIT_std,buro_DAYS_CREDIT_min,buro_DAYS_CREDIT_max,buro_CREDIT_TYPE_Mobile operator loan_mean,buro_CREDIT_TYPE_Mobile operator loan_std,buro_CREDIT_TYPE_Mobile operator loan_min,buro_CREDIT_TYPE_Mobile operator loan_max,buro_CREDIT_TYPE_Unknown type of loan_mean,buro_CREDIT_TYPE_Unknown type of loan_std,buro_CREDIT_TYPE_Unknown type of loan_min,buro_CREDIT_TYPE_Unknown type of loan_max,buro_STATUS_2_mean,buro_STATUS_2_std,buro_STATUS_2_min,buro_STATUS_2_max,buro_CREDIT_TYPE_Loan for purchase of shares (margin lending)_mean,buro_CREDIT_TYPE_Loan for purchase of shares (margin lending)_std,buro_CREDIT_TYPE_Loan for purchase of shares (margin lending)_min,buro_CREDIT_TYPE_Loan for purchase of shares (margin lending)_max,buro_CREDIT_DAY_OVERDUE_mean,buro_CREDIT_DAY_OVER

In [ ]:
# partitioning
train = appl[appl["SK_ID_CURR"].isin(y["SK_ID_CURR"]) == True]
test  = appl[appl["SK_ID_CURR"].isin(y["SK_ID_CURR"]) == False]
#del appl

In [ ]:
# check dimensions
print(train.shape)
print(test.shape)
train.head()

(307511, 1344)
(48744, 1344)


,SK_ID_CURR,app_CNT_CHILDREN,app_AMT_INCOME_TOTAL,app_AMT_CREDIT,app_AMT_ANNUITY,app_AMT_GOODS_PRICE,app_REGION_POPULATION_RELATIVE,app_DAYS_BIRTH,app_DAYS_EMPLOYED,app_DAYS_REGISTRATION,app_DAYS_ID_PUBLISH,app_OWN_CAR_AGE,app_FLAG_MOBIL,app_FLAG_EMP_PHONE,app_FLAG_WORK_PHONE,app_FLAG_CONT_MOBILE,app_FLAG_PHONE,app_FLAG_EMAIL,app_CNT_FAM_MEMBERS,app_REGION_RATING_CLIENT,app_REGION_RATING_CLIENT_W_CITY,app_HOUR_APPR_PROCESS_START,app_REG_REGION_NOT_LIVE_REGION,app_REG_REGION_NOT_WORK_REGION,app_LIVE_REGION_NOT_WORK_REGION,app_REG_CITY_NOT_LIVE_CITY,app_REG_CITY_NOT_WORK_CITY,app_LIVE_CITY_NOT_WORK_CITY,app_EXT_SOURCE_1,app_EXT_SOURCE_2,app_EXT_SOURCE_3,app_APARTMENTS_AVG,app_BASEMENTAREA_AVG,app_YEARS_BEGINEXPLUATATION_AVG,app_YEARS_BUILD_AVG,app_COMMONAREA_AVG,app_ELEVATORS_AVG,app_ENTRANCES_AVG,app_FLOORSMAX_AVG,app_FLOORSMIN_AVG,app_LANDAREA_AVG,app_LIVINGAPARTMENTS_AVG,app_LIVINGAREA_AVG,app_NONLIVINGAPARTMENTS_AVG,app_NONLIVINGAREA_AVG,app_YEARS_BUILD_MODE,app_OBS_30_CNT_SOCIAL_CIRCLE,app_DEF_30_CNT_SOCIAL_CIRCLE,app_OBS_60_CNT_SOCIAL_CIRCLE,app_DEF_60_CNT_SOCIAL_CIRCLE,app_DAYS_LAST_PHONE_CHANGE,app_FLAG_DOCUMENT_2,app_FLAG_DOCUMENT_3,app_FLAG_DOCUMENT_4,app_FLAG_DOCUMENT_5,app_FLAG_DOCUMENT_6,app_FLAG_DOCUMENT_7,app_FLAG_DOCUMENT_8,app_FLAG_DOCUMENT_9,app_FLAG_DOCUMENT_10,app_FLAG_DOCUMENT_11,app_FLAG_DOCUMENT_12,app_FLAG_DOCUMENT_13,app_FLAG_DOCUMENT_14,app_FLAG_DOCUMENT_15,app_FLAG_DOCUMENT_16,app_FLAG_DOCUMENT_17,app_FLAG_DOCUMENT_18,app_FLAG_DOCUMENT_19,app_FLAG_DOCUMENT_20,app_FLAG_DOCUMENT_21,app_AMT_REQ_CREDIT_BUREAU_HOUR,app_AMT_REQ_CREDIT_BUREAU_DAY,app_AMT_REQ_CREDIT_BUREAU_WEEK,app_AMT_REQ_CREDIT_BUREAU_MON,app_AMT_REQ_CREDIT_BUREAU_QRT,app_AMT_REQ_CREDIT_BUREAU_YEAR,app_CREDIT_BY_INCOME,app_ANNUITY_BY_INCOME,app_GOODS_PRICE_BY_INCOME,app_INCOME_PER_PERSON,app_PERCENT_WORKED,app_CNT_ADULTS,app_CHILDREN_RATIO,app_ANNUITY LENGTH,app_EXT_SOURCE_MEAN,app_NUM_EXT_SOURCES,app_NUM_DOCUMENTS,app_OWN_CAR_AGE_RATIO,app_DAYS_ID_PUBLISHED_RATIO,app_DAYS_REGISTRATION_RATIO,app_DAYS_LAST_PHONE_CHANGE_RATIO,index_x,buro_CNT_BURO_CLOSED_mean,buro_CNT_BURO_CLOSED_std,buro_CNT_BURO_CLOSED_min,buro_CNT_BURO_CLOSED_max,buro_CREDIT_TYPE_Consumer credit_mean,buro_CREDIT_TYPE_Consumer credit_std,buro_CREDIT_TYPE_Consumer credit_min,buro_CREDIT_TYPE_Consumer credit_max,buro_STATUS_1_mean,buro_STATUS_1_std,buro_STATUS_1_min,buro_STATUS_1_max,buro_CREDIT_CURRENCY_currency 4_mean,buro_CREDIT_CURRENCY_currency 4_std,buro_CREDIT_CURRENCY_currency 4_min,buro_CREDIT_CURRENCY_currency 4_max,buro_CREDIT_TYPE_Real estate loan_mean,buro_CREDIT_TYPE_Real estate loan_std,buro_CREDIT_TYPE_Real estate loan_min,buro_CREDIT_TYPE_Real estate loan_max,buro_AMT_CREDIT_SUM_mean,buro_AMT_CREDIT_SUM_std,buro_AMT_CREDIT_SUM_min,buro_AMT_CREDIT_SUM_max,buro_CREDIT_TYPE_Credit card_mean,buro_CREDIT_TYPE_Credit card_std,buro_CREDIT_TYPE_Credit card_min,buro_CREDIT_TYPE_Credit card_max,buro_AMT_MAX_OVERDUE_RATIO_1_mean,buro_AMT_MAX_OVERDUE_RATIO_1_std,buro_AMT_MAX_OVERDUE_RATIO_1_min,buro_AMT_MAX_OVERDUE_RATIO_1_max,buro_CREDIT_TYPE_Interbank credit_mean,buro_CREDIT_TYPE_Interbank credit_std,buro_CREDIT_TYPE_Interbank credit_min,buro_CREDIT_TYPE_Interbank credit_max,buro_DAYS_CREDIT_mean,buro_DAYS_CREDIT_std,buro_DAYS_CREDIT_min,buro_DAYS_CREDIT_max,buro_CREDIT_TYPE_Mobile operator loan_mean,buro_CREDIT_TYPE_Mobile operator loan_std,buro_CREDIT_TYPE_Mobile operator loan_min,buro_CREDIT_TYPE_Mobile operator loan_max,buro_CREDIT_TYPE_Unknown type of loan_mean,buro_CREDIT_TYPE_Unknown type of loan_std,buro_CREDIT_TYPE_Unknown type of loan_min,buro_CREDIT_TYPE_Unknown type of loan_max,buro_STATUS_2_mean,buro_STATUS_2_std,buro_STATUS_2_min,buro_STATUS_2_max,buro_CREDIT_TYPE_Loan for purchase of shares (margin lending)_mean,buro_CREDIT_TYPE_Loan for purchase of shares (margin lending)_std,buro_CREDIT_TYPE_Loan for purchase of shares (margin lending)_min,buro_CREDIT_TYPE_Loan for purchase of shares (margin lending)_max,buro_CREDIT_DAY_OVERDUE_mean,buro_CREDIT_DAY_OVER

In [ ]:
# export CSV
train.to_csv("../data/prepared/train_full_cor_2025.csv", index = False, float_format = "%.8f")
test.to_csv("../data/prepared/test_full_cor_2025.csv",   index = False, float_format = "%.8f")
y.to_csv("../data/prepared/y_full_cor_2025.csv",         index = False, float_format = "%.8f")